<a href="https://colab.research.google.com/github/tousifo/ml_notebooks/blob/main/Final_Blend_DermaMNIST_V28_LATENT_VQC_INPUT_BLEND_FINAL_COLAB_PATCHED_V3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Blend / DermaMNIST V28 — Latent VQC-Input Blend Final

This notebook implements the V28 two-branch design:

1. **Input-space Blend boundary:** V23–V27 are retained as negative evidence because visual/input-space triggers pass through the ResNet context bottleneck before reaching the VQC.
2. **VQC-input-space Blend branch:** a separate white-box latent trigger is injected after the compressor and before the VQC, so raw pixels and ResNet context remain unchanged while VQC measurements shift.

No input-space trigger search is run here. No hue, random, semantic, DC-shift, low-contrast, or sinusoidal trigger is used as the final positive path.


# V28 Design Memory

1. V23–V27 input-space Blend did not produce QNN/QMRS superiority.
2. The reason is the ResNet context bottleneck: every pixel-space trigger must be processed by ResNet before the VQC receives compressed quantum input.
3. Input-space Blend is retained as a negative boundary branch.
4. V28 implements **latent / VQC-input-space Blend** as a separate mechanistic branch.
5. This is **not** the same threat model as input-space Blend.
6. The claim must be labelled honestly as a white-box latent/VQC-input attack branch.
7. Lane A = matched unsupervised comparison.
8. Lane B = supervised audit only.
9. QXAI must show measurement/observable shift.


**Colab V2 integration note:** dataset stratification helper returns `(X, y, keep_idx)`; V28 latent poison construction now consumes only the first two tensors and ignores the index safely.

## V28 Colab Patch V3 — Tensor Contiguity Fix

This patch fixes a PyTorch runtime issue in V28 evaluation where non-contiguous tensor slices were flattened with `.view(...)`. The VQC-input Blend methodology is unchanged.

Patch applied:
- Replace unsafe tensor `.view(...)` calls with `.reshape(...)`.
- Preserve latent/VQC-input trigger branch.
- Preserve raw/context zero-shift gates.
- Preserve Lane A/B split and score-direction verification.
- Preserve no-HSV, no-sinusoidal-final-trigger, no-low-contrast restrictions.


## V28 Colab Integration Patch

This patched notebook fixes a dataset-bundle compatibility issue: older loader cells returned `class_map`, `X_train`, and `X_val`, while the V28 latent branch expects `label_map`, `train`, and `val` aliases. The patch normalizes these aliases without changing the V28 threat model, trigger mechanism, Lane A/B gates, or metrics.


# V28 Logic Verification

| Question | Answer |
|---|---|
| What failed in V23–V27? | Input-space triggers produced valid attacks but failed quantum-preferential detection. QPR stayed below the required gate and classical context/raw baselines were at least as strong as QMRS. |
| Why input-space Blend is blocked? | Pixel triggers enter through the ResNet-18 context representation before the VQC. Classical detectors observe the perturbation before the 512→8 compression bottleneck. |
| What is the V28 solution? | Keep input-space Blend as a negative boundary and add a separate VQC-input-space branch where the trigger is injected directly into the 8-dimensional quantum input after the compressor. |
| Is V28 still input-space Blend? | No. It is a white-box latent/VQC-input Blend branch and must not be labelled as ordinary input-space Blend. |
| How should it be labelled honestly? | “VQC-Input Blend: a white-box latent backdoor injected into the 8-dimensional VQC input space after classical feature extraction.” |
| What is the exact injection point? | After `q_input = model.vqc.compress(ctx)` and before the VQC qnode / measurement layer. |
| What metrics decide success? | ASR, CA, near-zero pixel/context delta, nonzero measurement shift, QMRS Lane A AUPRC/F1 beating raw/context Lane A baselines, plus non-empty QXAI observable shift. |


In [ ]:
# ============================================================
# Imports, configuration, reproducibility — Blend / DermaMNIST
# ============================================================

import os, sys, json, math, random, subprocess
from dataclasses import dataclass, asdict
from typing import Optional, Tuple
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import torchvision.transforms as T
from torchvision import models
from torchvision.models import ResNet18_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, average_precision_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import FastICA, PCA
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

INSTALL_MISSING_PACKAGES = True

def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        if not INSTALL_MISSING_PACKAGES:
            raise
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])
        return __import__(import_name)

medmnist = ensure_package("medmnist")
from medmnist import INFO
qml = ensure_package("pennylane")

ATTACK_NAME = "blend"
DATASET_NAME = "dermamnist"
RUN_DERMAMNIST_BLEND = True
RUN_PATHMNIST_FIBA = False
RUN_QTROJAN = False
RUN_CLASSICAL_POISONED_BASELINE = False
RUN_PAIR_SELECTION = False
RUN_QSENTRY_USEFULNESS_ABLATION = False
RUN_SUPERIORITY_MODULES = False
RUN_POISON_RATE_TRAINING_SWEEP = False
RUN_STEALTH_STRENGTH_SWEEP = False
RUN_QXAI_BLEND_EXPLANATION = False
RUN_BLEND_DATA_SANITIZATION = False

DEFAULT_OUT_DIR = "/kaggle/working/outputs_blend_dermamnist" if os.path.exists("/kaggle/working") else "/mnt/data/outputs_blend_dermamnist" if os.path.exists("/mnt/data") else "./outputs_blend_dermamnist"

CANDIDATE_PAIRS_BLEND = [(0, 4), (1, 4), (2, 4), (5, 4), (6, 4)]

@dataclass
class QMedShieldBlendConfig:
    primary_seed: int = 42
    dataset_name: str = DATASET_NAME
    attack_type: str = ATTACK_NAME
    n_classes: int = 7
    native_image_size: int = 28
    model_image_size: int = 96
    source_class: Optional[int] = None
    target_class: Optional[int] = None
    max_train_n: int = 5000
    pilot_max_train_n: int = 1500
    max_clean_test_n: int = 1200
    max_asr_n: int = 800
    val_size: float = 0.15
    use_imagenet_weights: bool = True
    freeze_resnet_lower_blocks: bool = True
    n_qubits: int = 8
    vqc_layers: int = 4
    obs_mode: str = "ZX_ALT"
    train_batch_size: int = 48
    eval_batch_size: int = 96
    clean_epochs: int = 8
    pilot_epochs: int = 2
    lr_base: float = 5e-4
    weight_decay: float = 1e-4
    label_smoothing: float = 0.10
    clean_acc_min: float = 0.50
    asr_min: float = 0.80
    strict_attack_gate: bool = True
    ica_components: int = 4
    kmeans_n_init: int = 10
    fpr_target: float = 0.05
    k_values: Tuple[int, ...] = (3, 5, 8, 10)
    threshold_method: str = "clean_val_95pct"
    clean_source_n: int = 450
    clean_target_n: int = 500
    poison_n: int = 50
    blend_alpha: float = 0.20
    blend_alpha_values: Tuple[float, ...] = (0.20, 0.15, 0.10, 0.07, 0.05)
    blend_poison_rate: float = 0.10
    qsentry_poison_ratios: Tuple[float, ...] = (0.01, 0.05, 0.10)
    superiority_poison_rates: Tuple[float, ...] = (0.01, 0.05, 0.10)
    stealth_alpha_values: Tuple[float, ...] = (0.20, 0.15, 0.10, 0.07, 0.05)
    superiority_sweep_epochs: int = 8
    sanitization_epochs: int = 2
    # Active probe logic: evaluate how model measurements change when a suspected Blend trigger is re-applied.
    # This is not test tuning; alphas are fixed before evaluation and used for both validation and test pools.
    active_probe_alphas: Tuple[float, ...] = (0.03, 0.07, 0.15)
    qnn_ensemble_top_m_values: Tuple[int, ...] = (1, 2, 3, 5, 999)
    qsentry_threshold_methods: Tuple[str, ...] = ("top_expected_poison_count", "clean_val_95pct")
    out_dir: str = DEFAULT_OUT_DIR

cfg = QMedShieldBlendConfig()
os.makedirs(cfg.out_dir, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NEEDS_VERIFICATION = []
PLANNED_PENDING = []

assert ATTACK_NAME == "blend"
assert DATASET_NAME == "dermamnist"
assert RUN_DERMAMNIST_BLEND is True
assert RUN_PATHMNIST_FIBA is False
assert RUN_QTROJAN is False
assert cfg.attack_type == "blend" and cfg.dataset_name == "dermamnist"


def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_all_seeds(cfg.primary_seed)
print("Device:", DEVICE)
print("Output directory:", cfg.out_dir)
display(pd.DataFrame([asdict(cfg)]).T.rename(columns={0: "value"}))


# Colab-safe split controls
FREEZE_BLEND_CONFIG = True
RUN_PAIR_SEARCH = False
RUN_25_PILOT_SEARCH = False
RUN_CLASSICAL_BASELINE_IN_NOTEBOOK1 = False
RUN_QXAI_IN_NOTEBOOK1 = False
RUN_SANITIZATION = False
SAVE_FEATURE_TENSORS = True
LOAD_FEATURE_TENSORS_IF_AVAILABLE = True
SKIP_COMPLETED_STAGES = True
RUN_TRIGGER_TYPE_VALIDATION = False  # optional validation-only diagnostic, disabled by default

# Frozen validated Blend configuration. Do not rerun pair search unless explicitly enabled.
cfg.source_class = 5
cfg.target_class = 4
cfg.blend_alpha = 0.07
cfg.blend_poison_rate = 0.10
assert cfg.source_class == 5 and cfg.target_class == 4
assert abs(cfg.blend_alpha - 0.07) < 1e-12
assert abs(cfg.blend_poison_rate - 0.10) < 1e-12


# Colab ZIP export controls
# After Notebook 1 finishes, it zips outputs_blend_dermamnist/ and downloads it.
# Upload this ZIP into Notebook 2 when asked.
AUTO_ZIP_AND_DOWNLOAD_OUTPUTS = True
OUTPUT_ZIP_NAME = "outputs_blend_dermamnist.zip"
OUTPUT_ZIP_PATH = os.path.abspath(OUTPUT_ZIP_NAME)


Device: cpu
Output directory: ./outputs_blend_dermamnist


,value
primary_seed,42
dataset_name,dermamnist
attack_type,blend
n_classes,7
native_image_size,28
model_image_size,96
source_class,None
target_class,None
max_train_n,5000
pilot_max_train_n,1500


In [ ]:
# ============================================================
# Dataset loading utilities
# ============================================================

def load_medmnist_split(dataset_name: str, split: str, download: bool = True):
    info = INFO[dataset_name]
    DataClass = getattr(medmnist, info["python_class"])
    ds = DataClass(split=split, download=download)
    X = torch.tensor(ds.imgs, dtype=torch.float32)
    if X.ndim == 3:
        X = X.unsqueeze(-1)
    X = X.permute(0, 3, 1, 2) / 255.0
    if X.shape[1] == 1:
        X = X.repeat(1, 3, 1, 1)
    y = torch.tensor(ds.labels.squeeze(), dtype=torch.long)
    return X, y, info


def label_map_to_int(label_map):
    out = {}
    for k, v in label_map.items():
        name = v[0] if isinstance(v, (list, tuple)) else str(v)
        out[int(k)] = name
    return out


def load_dataset_bundle(dataset_name: str):
    X_train, y_train, info = load_medmnist_split(dataset_name, "train")
    X_val, y_val, _ = load_medmnist_split(dataset_name, "val")
    X_test, y_test, _ = load_medmnist_split(dataset_name, "test")
    class_map = label_map_to_int(info["label"])
    print(f"Loaded {dataset_name}: train={X_train.shape}, val={X_val.shape}, test={X_test.shape}")
    display(pd.DataFrame([{"class_id": k, "class_name": v} for k, v in class_map.items()]))
    # V28 compatibility aliases: newer cells expect tuple-style split keys and label_map.
    return {
        "X_train": X_train, "y_train": y_train,
        "X_val": X_val, "y_val": y_val,
        "X_test": X_test, "y_test": y_test,
        "train": (X_train, y_train),
        "val": (X_val, y_val),
        "test": (X_test, y_test),
        "class_map": class_map,
        "label_map": class_map,
        "info": info,
    }


def resize_tensor_images(X, size):
    if X.shape[-1] == size and X.shape[-2] == size:
        return X.float()
    return F.interpolate(X.float(), size=(size, size), mode="bilinear", align_corners=False)


def clone_cfg(base_cfg, **updates):
    data = asdict(base_cfg)
    data.update(updates)
    return type(base_cfg)(**data)


In [ ]:
# ============================================================
# Model architecture and training helpers
# ============================================================

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).reshape(1,3,1,1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).reshape(1,3,1,1)


def preprocess_for_resnet(x, model_image_size):
    x = F.interpolate(x, size=(model_image_size, model_image_size), mode="bilinear", align_corners=False)
    return (x - IMAGENET_MEAN.to(x.device)) / IMAGENET_STD.to(x.device)


def measurement_dim(n_qubits, obs_mode):
    obs = obs_mode.upper().strip()
    if obs in ["Z_ONLY", "ZX_ALT"]:
        return n_qubits
    if obs == "XYZ":
        return 3 * n_qubits
    raise ValueError(f"Unknown obs_mode={obs_mode}")


def make_vqc_qnode(n_qubits, n_layers, obs_mode="ZX_ALT"):
    try:
        dev = qml.device("lightning.qubit", wires=n_qubits)
    except Exception:
        print("[warning] lightning.qubit unavailable; using default.qubit")
        dev = qml.device("default.qubit", wires=n_qubits)
    obs = obs_mode.upper().strip()

    @qml.qnode(dev, interface="torch", diff_method="best")
    def circuit(angles, weights, input_scales, input_bias, trojan_angles, trigger_flag):
        for layer in range(n_layers):
            for q in range(n_qubits):
                qml.Rot(weights[layer, q, 0], weights[layer, q, 1], weights[layer, q, 2], wires=q)
            for q in range(n_qubits):
                qml.RY(input_scales[layer, q] * angles[q] + input_bias[layer, q], wires=q)
            # No QTrojan is used in this notebook; trigger_flag stays 0. This parameter remains only because the shared VQC layer supports QTrojan notebooks.
            for q in range(n_qubits):
                qml.RY(trigger_flag * trojan_angles[layer, q], wires=q)
            if layer % 2 == 0:
                for q in range(n_qubits):
                    qml.CNOT(wires=[q, (q + 1) % n_qubits])
            else:
                for q in range(n_qubits):
                    qml.CNOT(wires=[(q + 1) % n_qubits, q])
        if obs == "Z_ONLY":
            return [qml.expval(qml.PauliZ(q)) for q in range(n_qubits)]
        if obs == "XYZ":
            out = []
            for q in range(n_qubits):
                out.extend([qml.expval(qml.PauliX(q)), qml.expval(qml.PauliY(q)), qml.expval(qml.PauliZ(q))])
            return out
        return [qml.expval(qml.PauliZ(q)) if q % 2 == 0 else qml.expval(qml.PauliX(q)) for q in range(n_qubits)]
    return circuit


class StrongVQCMeasurementLayer(nn.Module):
    def __init__(self, in_dim, n_qubits, n_layers, obs_mode, trojan_init_std=0.0):
        super().__init__()
        self.n_qubits = n_qubits
        self.n_layers = n_layers
        self.obs_mode = obs_mode
        self.compress = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, 32),
            nn.Tanh(),
            nn.Linear(32, n_qubits),
            nn.Tanh(),
        )
        self.weights = nn.Parameter(0.01 * torch.randn(n_layers, n_qubits, 3))
        self.input_scales = nn.Parameter(torch.ones(n_layers, n_qubits))
        self.input_bias = nn.Parameter(torch.zeros(n_layers, n_qubits))
        self.trojan_angles = nn.Parameter(trojan_init_std * torch.randn(n_layers, n_qubits))
        self.qnode = make_vqc_qnode(n_qubits, n_layers, obs_mode)

    def forward(self, context, trojan_mask=None):
        angles = math.pi * (self.compress(context) + 1.0) / 2.0
        if trojan_mask is None:
            trojan_mask = torch.zeros(angles.shape[0], device=angles.device, dtype=angles.dtype)
        else:
            trojan_mask = trojan_mask.to(device=angles.device, dtype=angles.dtype).reshape(-1)
        vals = [
            torch.stack(self.qnode(a, self.weights, self.input_scales, self.input_bias, self.trojan_angles, flag)).float()
            for a, flag in zip(angles, trojan_mask)
        ]
        return torch.stack(vals, dim=0)


class QMedShieldHybridQNN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        weights = None
        if cfg.use_imagenet_weights:
            try:
                weights = ResNet18_Weights.IMAGENET1K_V1
            except Exception:
                weights = None
        try:
            resnet = models.resnet18(weights=weights)
        except Exception as exc:
            msg = f"needs verification: ImageNet weights unavailable ({exc}); using random ResNet-18 init."
            print("[warning]", msg)
            NEEDS_VERIFICATION.append(msg)
            resnet = models.resnet18(weights=None)
        self.stem = nn.Sequential(*list(resnet.children())[:-2])
        if cfg.freeze_resnet_lower_blocks:
            for p in self.stem[:6].parameters():
                p.requires_grad = False
        self.rnn = nn.GRU(input_size=512, hidden_size=256, num_layers=2, batch_first=True, bidirectional=True, dropout=0.2)
        self.vqc = StrongVQCMeasurementLayer(512, cfg.n_qubits, cfg.vqc_layers, cfg.obs_mode, 0.0)
        self.q_head = nn.Linear(measurement_dim(cfg.n_qubits, cfg.obs_mode), cfg.n_classes)

    def _context(self, x_preprocessed):
        feats = self.stem(x_preprocessed)
        B, C, H, W = feats.shape
        seq = feats.reshape(B, C, H*W).permute(0, 2, 1)
        out, _ = self.rnn(seq)
        return out.mean(dim=1)

    def forward(self, x_preprocessed, t=0.0, return_measurements=False):
        ctx = self._context(x_preprocessed)
        mask = torch.zeros(ctx.shape[0], device=ctx.device, dtype=ctx.dtype)
        qfeat = self.vqc(ctx, trojan_mask=mask)
        logits = self.q_head(qfeat)
        if return_measurements:
            return logits, qfeat, ctx
        return logits


class ClassicalResNetBiGRU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        weights = None
        if cfg.use_imagenet_weights:
            try:
                weights = ResNet18_Weights.IMAGENET1K_V1
            except Exception:
                weights = None
        try:
            resnet = models.resnet18(weights=weights)
        except Exception:
            resnet = models.resnet18(weights=None)
        self.stem = nn.Sequential(*list(resnet.children())[:-2])
        if cfg.freeze_resnet_lower_blocks:
            for p in self.stem[:6].parameters():
                p.requires_grad = False
        self.rnn = nn.GRU(input_size=512, hidden_size=256, num_layers=2, batch_first=True, bidirectional=True, dropout=0.2)
        self.head = nn.Linear(512, cfg.n_classes)

    def _context(self, x_preprocessed):
        feats = self.stem(x_preprocessed)
        B, C, H, W = feats.shape
        seq = feats.reshape(B, C, H*W).permute(0, 2, 1)
        out, _ = self.rnn(seq)
        return out.mean(dim=1)

    def forward(self, x_preprocessed, return_context=False):
        ctx = self._context(x_preprocessed)
        logits = self.head(ctx)
        if return_context:
            return logits, ctx
        return logits


def verify_architecture(model, cfg, device):
    model = model.to(device)
    dummy = torch.rand(4, 3, cfg.native_image_size, cfg.native_image_size, device=device)
    xp = preprocess_for_resnet(dummy, cfg.model_image_size)
    logits, qfeat, ctx = model(xp, return_measurements=True)
    assert logits.shape == (4, cfg.n_classes), logits.shape
    assert qfeat.shape == (4, measurement_dim(cfg.n_qubits, cfg.obs_mode)), qfeat.shape
    assert ctx.shape == (4, 512), ctx.shape
    print("Architecture verified:", logits.shape, qfeat.shape, ctx.shape)


def stratified_limit_tensors(X, y, max_n, seed):
    if max_n is None or len(y) <= max_n:
        return X, y, np.arange(len(y))
    idx = np.arange(len(y))
    _, keep = train_test_split(idx, test_size=max_n, random_state=seed, stratify=y.cpu().numpy())
    keep = np.sort(keep)
    return X[keep], y[keep], keep


def make_loader(X, y, batch_size, shuffle):
    return DataLoader(TensorDataset(X.float(), y.long()), batch_size=batch_size, shuffle=shuffle, num_workers=0)


def optimizer_for_qnn(model, cfg):
    return torch.optim.AdamW([
        {"params": model.stem[6:].parameters(), "lr": cfg.lr_base * 0.1},
        {"params": model.rnn.parameters(), "lr": cfg.lr_base},
        {"params": model.vqc.parameters(), "lr": cfg.lr_base},
        {"params": model.q_head.parameters(), "lr": cfg.lr_base},
    ], weight_decay=cfg.weight_decay)


def optimizer_for_classical(model, cfg):
    return torch.optim.AdamW([
        {"params": model.stem[6:].parameters(), "lr": cfg.lr_base * 0.1},
        {"params": model.rnn.parameters(), "lr": cfg.lr_base},
        {"params": model.head.parameters(), "lr": cfg.lr_base},
    ], weight_decay=cfg.weight_decay)


@torch.no_grad()
def evaluate_classifier(model, X, y, cfg, device):
    model.eval()
    loader = make_loader(X, y, cfg.eval_batch_size, shuffle=False)
    preds, probs = [], []
    for xb, _ in loader:
        xp = preprocess_for_resnet(xb.to(device), cfg.model_image_size)
        logits = model(xp)
        prob = torch.softmax(logits, dim=1).cpu().numpy()
        preds.append(prob.argmax(axis=1)); probs.append(prob)
    preds = np.concatenate(preds); probs = np.concatenate(probs)
    y_np = y.cpu().numpy() if isinstance(y, torch.Tensor) else np.asarray(y)
    return {"accuracy": float(accuracy_score(y_np, preds)), "macro_f1": float(f1_score(y_np, preds, average="macro", zero_division=0)), "preds": preds, "probs": probs}


@torch.no_grad()
def evaluate_classical_classifier(model, X, y, cfg, device):
    model.eval()
    loader = make_loader(X, y, cfg.eval_batch_size, shuffle=False)
    preds, probs = [], []
    for xb, _ in loader:
        xp = preprocess_for_resnet(xb.to(device), cfg.model_image_size)
        logits = model(xp)
        prob = torch.softmax(logits, dim=1).cpu().numpy()
        preds.append(prob.argmax(axis=1)); probs.append(prob)
    preds = np.concatenate(preds); probs = np.concatenate(probs)
    y_np = y.cpu().numpy() if isinstance(y, torch.Tensor) else np.asarray(y)
    return {"accuracy": float(accuracy_score(y_np, preds)), "macro_f1": float(f1_score(y_np, preds, average="macro", zero_division=0)), "preds": preds, "probs": probs}


def train_clean_stage(model, train_loader, val_loader, cfg, device):
    model = model.to(device)
    opt = optimizer_for_qnn(model, cfg)
    hist = []
    print(f"QNN poisoned/clean training: {cfg.clean_epochs} epochs")
    for ep in range(1, cfg.clean_epochs + 1):
        model.train(); loss_sum, n = 0.0, 0
        for xb, yb in train_loader:
            xp = preprocess_for_resnet(xb.to(device), cfg.model_image_size); yb = yb.to(device)
            opt.zero_grad(); logits = model(xp); loss = F.cross_entropy(logits, yb, label_smoothing=cfg.label_smoothing)
            loss.backward(); opt.step()
            loss_sum += float(loss.detach().cpu()) * len(xb); n += len(xb)
        val = evaluate_classifier(model, val_loader.dataset.tensors[0], val_loader.dataset.tensors[1], cfg, device)
        row = {"epoch": ep, "loss": loss_sum/max(n,1), "val_acc": val["accuracy"], "val_macro_f1": val["macro_f1"]}
        hist.append(row)
        print(f"Epoch {ep:02d}: loss={row['loss']:.4f}, val_CA={row['val_acc']:.4f}")
    return model, hist


def train_classical_stage(model, train_loader, val_loader, cfg, device, label="classical_poisoned"):
    model = model.to(device)
    opt = optimizer_for_classical(model, cfg)
    hist = []
    print(f"[{label}] training: {cfg.clean_epochs} epochs")
    for ep in range(1, cfg.clean_epochs + 1):
        model.train(); loss_sum, n = 0.0, 0
        for xb, yb in train_loader:
            xp = preprocess_for_resnet(xb.to(device), cfg.model_image_size); yb = yb.to(device)
            opt.zero_grad(); logits = model(xp); loss = F.cross_entropy(logits, yb, label_smoothing=cfg.label_smoothing)
            loss.backward(); opt.step()
            loss_sum += float(loss.detach().cpu()) * len(xb); n += len(xb)
        val = evaluate_classical_classifier(model, val_loader.dataset.tensors[0], val_loader.dataset.tensors[1], cfg, device)
        row = {"epoch": ep, "loss": loss_sum/max(n,1), "val_acc": val["accuracy"], "val_macro_f1": val["macro_f1"]}
        hist.append(row)
        print(f"[{label}] Epoch {ep:02d}: loss={row['loss']:.4f}, val_CA={row['val_acc']:.4f}")
    return model, hist


In [ ]:
# ============================================================
# Feature extraction, detector, ablation, plotting, and claim control
# ============================================================

@torch.no_grad()
def extract_qnn_features(model, X, cfg, device):
    model.eval(); q_rows, ctx_rows, prob_rows = [], [], []
    for s in range(0, len(X), cfg.eval_batch_size):
        xb = preprocess_for_resnet(X[s:s+cfg.eval_batch_size].to(device), cfg.model_image_size)
        logits, q, ctx = model(xb, return_measurements=True)
        q_rows.append(q.detach().cpu().numpy())
        ctx_rows.append(ctx.detach().cpu().numpy())
        prob_rows.append(torch.softmax(logits, dim=1).detach().cpu().numpy())
    return np.concatenate(q_rows), np.concatenate(ctx_rows), np.concatenate(prob_rows)


@torch.no_grad()
def extract_classical_context(model, X, cfg, device):
    if model is None:
        return None
    model.eval(); rows = []
    for s in range(0, len(X), cfg.eval_batch_size):
        xb = preprocess_for_resnet(X[s:s+cfg.eval_batch_size].to(device), cfg.model_image_size)
        _, ctx = model(xb, return_context=True)
        rows.append(ctx.detach().cpu().numpy())
    return np.concatenate(rows)



def sanitize_feature_matrix(X, name="feature_matrix"):
    """Return a finite float32 2-D feature matrix and record any repair in NEEDS_VERIFICATION."""
    X = np.asarray(X, dtype=np.float32)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    if X.ndim != 2:
        X = X.reshape(X.shape[0], -1)
    bad = ~np.isfinite(X)
    if bad.any():
        NEEDS_VERIFICATION.append(
            f"needs verification: sanitized {int(bad.sum())} non-finite values in {name}."
        )
        X = np.nan_to_num(X, nan=0.0, posinf=1e6, neginf=-1e6).astype(np.float32)
    return X


def quantum_readout_statistics(q):
    """Finite readout statistics for expectation values q in [-1, 1]."""
    q = sanitize_feature_matrix(q, "quantum_readout_input").astype(np.float64)
    q = np.clip(q, -1.0 + 1e-6, 1.0 - 1e-6)
    eps = 1e-6
    p = np.clip((q + 1.0) / 2.0, eps, 1.0 - eps)
    entropy = -(p * np.log(p) + (1.0 - p) * np.log(1.0 - p))
    saturation = np.abs(q)
    uncertainty = 1.0 - saturation
    out = np.concatenate([q, entropy, saturation, uncertainty], axis=1)
    return sanitize_feature_matrix(out, "quantum_readout_statistics")


def block_standardize(train_block, test_block):
    scaler = StandardScaler()
    train_z = scaler.fit_transform(train_block)
    test_z = scaler.transform(test_block)
    scale = np.sqrt(max(train_z.shape[1], 1))
    return train_z / scale, test_z / scale


def make_block_weighted_hybrid(val_blocks, test_blocks, weights=None):
    if weights is None:
        weights = {k: 1.0 for k in val_blocks}
    val_parts, test_parts = [], []
    for name in val_blocks:
        v, t = block_standardize(val_blocks[name], test_blocks[name])
        val_parts.append(weights.get(name, 1.0) * v)
        test_parts.append(weights.get(name, 1.0) * t)
    return np.concatenate(val_parts, axis=1), np.concatenate(test_parts, axis=1)



def detector_scores_from_features(X_val_feat, y_val, X_test_feat, cfg, k):
    """
    Robust QSentry-style anomaly scoring.

    Fixes implemented:
    - sanitizes NaN/Inf before scaling/reduction;
    - treats FastICA non-convergence as a real failure and falls back to PCA;
    - refuses invalid K values instead of silently producing broken rows.
    """
    from sklearn.exceptions import ConvergenceWarning
    import warnings

    X_val_feat = sanitize_feature_matrix(X_val_feat, "detector_val_features")
    X_test_feat = sanitize_feature_matrix(X_test_feat, "detector_test_features")
    y_val = np.asarray(y_val).astype(int)
    k = int(k)
    if k < 2:
        raise ValueError(f"K must be >=2, got {k}")
    if len(X_val_feat) < k:
        raise ValueError(f"Validation feature count {len(X_val_feat)} is smaller than K={k}")

    scaler = StandardScaler()
    Vv0 = scaler.fit_transform(X_val_feat)
    Vt0 = scaler.transform(X_test_feat)
    Vv0 = sanitize_feature_matrix(Vv0, "scaled_val_features")
    Vt0 = sanitize_feature_matrix(Vt0, "scaled_test_features")

    n_comp = min(int(cfg.ica_components), Vv0.shape[1], max(1, Vv0.shape[0] - 1))
    try:
        reducer = FastICA(
            n_components=n_comp,
            random_state=int(cfg.primary_seed),
            whiten="unit-variance",
            max_iter=3000,
            tol=1e-4,
        )
        with warnings.catch_warnings():
            warnings.filterwarnings("error", category=ConvergenceWarning)
            Vv = reducer.fit_transform(Vv0)
            Vt = reducer.transform(Vt0)
    except Exception as exc:
        NEEDS_VERIFICATION.append(
            f"needs verification: FastICA fallback to PCA for K={k}: {exc}"
        )
        reducer = PCA(n_components=n_comp, random_state=int(cfg.primary_seed))
        Vv = reducer.fit_transform(Vv0)
        Vt = reducer.transform(Vt0)

    Vv = sanitize_feature_matrix(Vv, "reduced_val_features")
    Vt = sanitize_feature_matrix(Vt, "reduced_test_features")

    km = KMeans(n_clusters=k, random_state=int(cfg.primary_seed), n_init=int(cfg.kmeans_n_init))
    val_cluster = km.fit_predict(Vv)
    centers = km.cluster_centers_
    rows = []
    for c in range(k):
        mask = val_cluster == c
        rows.append((c, float(y_val[mask].mean()) if mask.any() else 0.0, int(mask.sum())))
    suspicious_cluster = sorted(rows, key=lambda x: (-x[1], x[2]))[0][0]

    def score(V):
        d_susp = np.linalg.norm(V - centers[suspicious_cluster], axis=1)
        other = [j for j in range(k) if j != suspicious_cluster]
        d_other = np.min(np.stack([np.linalg.norm(V - centers[j], axis=1) for j in other], axis=1), axis=1)
        return sanitize_feature_matrix((d_other - d_susp).reshape(-1, 1), "detector_scores").ravel()

    return score(Vv), score(Vt), {"k": k, "suspicious_cluster": int(suspicious_cluster), "cluster_rows": rows}


def prediction_from_scores(val_scores, y_val, test_scores, expected_poison_count, method="clean_val_95pct", fpr=0.05):
    expected_poison_count = int(max(1, min(len(test_scores), round(expected_poison_count))))
    if method == "clean_val_95pct":
        clean_val_scores = val_scores[np.asarray(y_val) == 0]
        tau = float(np.percentile(clean_val_scores, 100 * (1 - fpr)))
        return (test_scores >= tau).astype(int), tau, "clean_val_95pct"
    if method not in ["top_expected_poison_count", "relative_cluster_size"]:
        raise ValueError(f"Unknown threshold method: {method}")
    pred = np.zeros(len(test_scores), dtype=int)
    top = np.argsort(test_scores)[-expected_poison_count:]
    pred[top] = 1
    return pred, None, method


def eval_detection(y_true, scores, pred):
    out = {
        "F1": float(f1_score(y_true, pred, zero_division=0)),
        "Precision": float(precision_score(y_true, pred, zero_division=0)),
        "Recall": float(recall_score(y_true, pred, zero_division=0)),
        "AUPRC": float(average_precision_score(y_true, scores)),
    }
    try:
        out["AUROC"] = float(roc_auc_score(y_true, scores))
    except Exception:
        out["AUROC"] = np.nan
    return out


def select_k_and_evaluate(feature_name, X_val_feat, y_val, X_test_feat, y_test, cfg, expected_poison_count, threshold_method=None):
    threshold_method = threshold_method or cfg.threshold_method
    val_rows, candidates = [], {}
    for k in cfg.k_values:
        try:
            val_scores, test_scores, meta = detector_scores_from_features(X_val_feat, y_val, X_test_feat, cfg, k)
            val_pred, val_tau, _ = prediction_from_scores(val_scores, y_val, val_scores, max(1, int(np.asarray(y_val).sum())), method=threshold_method, fpr=cfg.fpr_target)
            m_val = eval_detection(y_val, val_scores, val_pred)
            val_rows.append({"Feature Space": feature_name, "K": k, **m_val})
            candidates[k] = (val_scores, test_scores, meta)
        except Exception as e:
            val_rows.append({"Feature Space": feature_name, "K": k, "error": str(e)})
    val_df = pd.DataFrame(val_rows)
    valid = val_df.dropna(subset=["AUPRC", "F1"], how="any")
    if valid.empty:
        raise RuntimeError(f"No valid K for {feature_name}. Details: {val_df}")
    best_k = int(valid.sort_values(["AUPRC", "F1"], ascending=False).iloc[0]["K"])
    val_scores, test_scores, meta = candidates[best_k]
    test_pred, tau, threshold_name = prediction_from_scores(val_scores, y_val, test_scores, expected_poison_count, method=threshold_method, fpr=cfg.fpr_target)
    test_metrics = eval_detection(y_test, test_scores, test_pred)
    row = {"Feature Space": feature_name, "K": best_k, "Threshold Method": threshold_name, "Tau": tau, **test_metrics}
    return row, val_df, test_scores


def plot_detection_score_distribution(scores, y_true, title, out_path):
    y_true = np.asarray(y_true)
    plt.figure(figsize=(7, 4))
    plt.hist(scores[y_true == 0], bins=40, alpha=0.65, label="clean")
    plt.hist(scores[y_true == 1], bins=40, alpha=0.65, label="poison")
    plt.title(title)
    plt.xlabel("Detection score")
    plt.ylabel("Count")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()


def save_claim_decision(detection_df, attack_name, dataset_name, out_path):
    if detection_df.empty or "Feature Space" not in detection_df.columns:
        claim = "pending experiment"
        reason = "Detection table is empty or attack gate did not pass."
    else:
        def metric_for(name, metric):
            rows = detection_df[detection_df["Feature Space"] == name]
            if rows.empty:
                return np.nan
            return float(rows.iloc[0][metric])
        q_static_f1, q_static_auprc = metric_for("Quantum static measurement", "F1"), metric_for("Quantum static measurement", "AUPRC")
        q_delta_f1, q_delta_auprc = metric_for("Quantum probe-delta measurement", "F1"), metric_for("Quantum probe-delta measurement", "AUPRC")
        classical_f1, classical_auprc = metric_for("Classical context baseline", "F1"), metric_for("Classical context baseline", "AUPRC")
        qnnctx_f1, qnnctx_auprc = metric_for("QNN context baseline", "F1"), metric_for("QNN context baseline", "AUPRC")
        bal_f1, bal_auprc = metric_for("Balanced quantum+context hybrid", "F1"), metric_for("Balanced quantum+context hybrid", "AUPRC")
        context_f1 = np.nanmax([classical_f1, qnnctx_f1])
        context_auprc = np.nanmax([classical_auprc, qnnctx_auprc])
        quantum_best_f1 = np.nanmax([q_static_f1, q_delta_f1])
        quantum_best_auprc = np.nanmax([q_static_auprc, q_delta_auprc])
        if quantum_best_f1 > context_f1 and quantum_best_auprc > context_auprc:
            claim = "Strong"
            reason = "Quantum-only or quantum-delta features outperform classical/context features in both F1 and AUPRC."
        elif bal_f1 > context_f1 and bal_auprc > context_auprc:
            claim = "Moderate"
            reason = "Balanced quantum+context hybrid improves beyond classical/context alone."
        elif quantum_best_f1 > 0.30 or quantum_best_auprc > 0.30:
            claim = "Weak"
            reason = "Quantum features provide useful signal, but not primary detection power."
        else:
            claim = "Unsupported"
            reason = "Classical/context features carry most of the detection performance."
    text = f"""# Claim decision — {attack_name} / {dataset_name}\n\nFinal claim category: **{claim}**\n\nReason: {reason}\n\nSafe wording: Do not claim quantum-measurement dominance unless the feature ablation table proves it. Use the table-generated category above.\n"""
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(text)
    print(text)
    return claim, reason


def save_qsentry_usefulness_decision(qsentry_df, out_path):
    """
    Conservative, table-driven decision for whether the Blend branch shows QSentry-style usefulness.
    This does not force quantum superiority. It compares raw, classical-context, QNN-context,
    quantum-only, and hybrid features at each poison ratio under top-expected-poison thresholding.
    """
    if qsentry_df is None or qsentry_df.empty:
        text = "# QSentry-style Blend usefulness decision\n\nStatus: **pending experiment**\n\nReason: ablation table is empty.\n"
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(text)
        print(text)
        return "pending experiment"

    lines = [
        "# QSentry-style Blend usefulness decision",
        "",
        "Rule: evaluate poison rates 1%, 5%, and 10% using validation-selected K and top-expected-poison thresholding.",
        "Do not claim QNN/quantum dominance unless the table shows it.",
        "",
    ]

    top_df = qsentry_df[qsentry_df["Threshold Method"] == "top_expected_poison_count"].copy()
    decisions = []

    for ratio in sorted(top_df["Target Poison Ratio"].dropna().unique()):
        sub = top_df[np.isclose(top_df["Target Poison Ratio"], ratio)].copy()
        if sub.empty:
            continue

        def metric(name, m):
            r = sub[sub["Feature Space"] == name]
            if r.empty or m not in r.columns:
                return np.nan
            return float(r.iloc[0][m])

        raw_f1 = metric("Raw pixel baseline", "F1")
        classical_f1 = metric("Classical context baseline", "F1")
        qnn_f1 = metric("QNN context baseline", "F1")
        qstatic_f1 = metric("Quantum static measurement", "F1")
        qdelta_f1 = metric("Quantum probe-delta measurement", "F1")
        hybrid_f1 = metric("Balanced quantum+context hybrid", "F1")

        classical_auprc = metric("Classical context baseline", "AUPRC")
        qnn_auprc = metric("QNN context baseline", "AUPRC")
        qstatic_auprc = metric("Quantum static measurement", "AUPRC")
        qdelta_auprc = metric("Quantum probe-delta measurement", "AUPRC")
        hybrid_auprc = metric("Balanced quantum+context hybrid", "AUPRC")

        quantum_best_f1 = np.nanmax([qstatic_f1, qdelta_f1])
        quantum_best_auprc = np.nanmax([qstatic_auprc, qdelta_auprc])
        context_best_f1 = np.nanmax([classical_f1, qnn_f1])
        context_best_auprc = np.nanmax([classical_auprc, qnn_auprc])

        raw_beaten = bool(np.nanmax([qnn_f1, qstatic_f1, qdelta_f1, hybrid_f1]) > raw_f1)
        classical_beaten_by_quantum = bool(quantum_best_f1 > classical_f1 and quantum_best_auprc >= classical_auprc)
        classical_beaten_by_hybrid = bool(hybrid_f1 > classical_f1 and hybrid_auprc >= classical_auprc)

        if classical_beaten_by_quantum:
            verdict = "strong quantum-only advantage"
        elif classical_beaten_by_hybrid:
            verdict = "moderate hybrid advantage"
        elif raw_beaten:
            verdict = "weak useful signal over raw baseline"
        else:
            verdict = "unsupported at this ratio"

        decisions.append(verdict)
        lines += [
            f"## Target poison ratio: {ratio:.0%}",
            f"- Raw F1: `{raw_f1:.4f}`",
            f"- Classical context F1/AUPRC: `{classical_f1:.4f}` / `{classical_auprc:.4f}`",
            f"- QNN context F1/AUPRC: `{qnn_f1:.4f}` / `{qnn_auprc:.4f}`",
            f"- Best quantum-only F1/AUPRC: `{quantum_best_f1:.4f}` / `{quantum_best_auprc:.4f}`",
            f"- Balanced hybrid F1/AUPRC: `{hybrid_f1:.4f}` / `{hybrid_auprc:.4f}`",
            f"- Verdict: **{verdict}**",
            "",
        ]

    if any("strong" in d for d in decisions):
        overall = "Strong"
        safe = "Quantum-only features outperform classical context for at least one poison-rate setting."
    elif any("moderate" in d for d in decisions):
        overall = "Moderate"
        safe = "Hybrid quantum+context features outperform classical context for at least one poison-rate setting."
    elif any("weak" in d for d in decisions):
        overall = "Weak"
        safe = "QNN/quantum features improve over raw pixel clustering but do not consistently beat classical context."
    else:
        overall = "Unsupported"
        safe = "The ablation does not show a reliable QNN/quantum advantage."

    lines += [
        "## Overall decision",
        f"Final category: **{overall}**",
        "",
        f"Safe paper wording: {safe}",
        "",
        "If the category is Weak or Unsupported, do not write that QNN/quantum detection is superior to classical context.",
    ]

    text = "\n".join(lines) + "\n"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(text)
    print(text)
    return overall


# Input-Space Blend Boundary Evidence

The notebook preserves the V23–V27 input-space Blend evidence as a negative boundary. These results are not rerun and are not used as a positive final branch.

| Version | Trigger | ASR | CA | QPR | QMRS | Classical | Decision |
|---|---|---:|---:|---:|---:|---:|---|
| V23 | random global Blend anchor | 0.9985 | 0.7556 | 0.2863 | 1.0000 | 1.0000 | Valid attack but classically obvious |
| V26 | hue/Qcolor-inspired input trigger | 0.9955 | 0.7307 | 0.1260 | 0.1972 | 1.0000 | Valid attack but worse quantum preference |
| V27 | proxy-optimized sinusoidal input trigger | 0.9568 | 0.7027 | 0.1256 | 0.1585 | 0.1879 | Valid attack but QPR and Lane A gates failed |

Conclusion: input-space Blend attacks were valid, but quantum-preferential detection failed. Therefore input-space Blend is not used as the positive QNN/QMRS branch.


In [ ]:

# ============================================================
# V28 input-space Blend negative-boundary summary export
# ============================================================
from pathlib import Path
import pandas as pd, json, os

V28_VERSION = "V28_LATENT_VQC_INPUT_BLEND_FINAL"
V28_BRANCH = "vqc_input_space_blend"
Path(cfg.out_dir).mkdir(parents=True, exist_ok=True)

boundary_rows = [
    {"Version": "V23", "Trigger": "random_global_blend_anchor", "ASR": 0.9985, "CA": 0.7556, "QPR": 0.2863,
     "QMRS": 1.0000, "Classical": 1.0000, "Decision": "valid_attack_but_classically_obvious"},
    {"Version": "V26", "Trigger": "hue_qcolor_input_trigger", "ASR": 0.9955, "CA": 0.7307, "QPR": 0.1260,
     "QMRS": 0.1972, "Classical": 1.0000, "Decision": "valid_attack_but_no_qmrs_win"},
    {"Version": "V27", "Trigger": "proxy_sinusoidal_input_trigger", "ASR": 0.9568, "CA": 0.7027, "QPR": 0.1256,
     "QMRS": 0.1585, "Classical": 0.1879, "Decision": "no_viable_quantum_preferential_trigger"},
]
boundary_df = pd.DataFrame(boundary_rows)
boundary_csv = Path(cfg.out_dir) / "blend_v28_input_space_boundary_summary.csv"
boundary_df.to_csv(boundary_csv, index=False)

boundary_md_path = Path(cfg.out_dir) / "blend_v28_input_space_boundary_summary.md"
boundary_md_path.write_text("""# V28 Input-Space Blend Boundary Summary

Input-space Blend attacks were valid, but quantum-preferential detection failed.
Therefore, input-space Blend is not used as the positive QNN/QMRS branch.

""" + boundary_df.to_markdown(index=False) + "\n", encoding="utf-8")
print("Wrote:", boundary_md_path)
display(boundary_df)


Wrote: outputs_blend_dermamnist/blend_v28_input_space_boundary_summary.md


,Version,Trigger,ASR,CA,QPR,QMRS,Classical,Decision
0,V23,random_global_blend_anchor,0.9985,0.7556,0.2863,1.0000,1.0000,valid_attack_but_classically_obvious
1,V26,hue_qcolor_input_trigger,0.9955,0.7307,0.1260,0.1972,1.0000,valid_attack_but_no_qmrs_win
2,V27,proxy_sinusoidal_input_trigger,0.9568,0.7027,0.1256,0.1585,0.1879,no_viable_quantum_preferential_trigger


In [ ]:

# ============================================================
# V28 FINAL: Latent / VQC-input-space Blend branch
# ============================================================
# Implements: vqc_input_space_blend
# Trigger injection point: after compressor, before VQC measurement circuit.
# This does not alter raw pixels or ResNet context features.

import os, json, math, time, zipfile, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, f1_score, silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

# -----------------------------
# V28 constants
# -----------------------------
SOURCE_CLASS = 5
TARGET_CLASS = 4
POISON_RATE = 0.10
ASR_GATE = 0.80
CA_MIN_THRESHOLD = max(0.60, float(getattr(cfg, "clean_acc_min", 0.50)))
PILOT_EPOCHS = 3
FULL_EPOCHS = 8
MAX_CANDIDATES = 16
SAVE_AFTER_EVERY_CANDIDATE = True
RESUME_FROM_CACHE = True
ALPHA_GRID = [0.25, 0.50, 0.75, 1.00]
V28_TRIGGER_CANDIDATES = [
    {"name": "basis_q0", "trigger": [1,0,0,0,0,0,0,0]},
    {"name": "basis_q1", "trigger": [0,1,0,0,0,0,0,0]},
    {"name": "alternating", "trigger": [1,-1,1,-1,1,-1,1,-1]},
    {"name": "learned_mean_shift", "trigger": "target_q_mean_minus_source_q_mean"},
]
EPS = 1e-8

cfg.source_class = SOURCE_CLASS
cfg.target_class = TARGET_CLASS
cfg.blend_poison_rate = POISON_RATE
cfg.pilot_epochs = PILOT_EPOCHS
set_all_seeds(cfg.primary_seed)
Path(cfg.out_dir).mkdir(parents=True, exist_ok=True)
(Path(cfg.out_dir) / "paper_figures").mkdir(parents=True, exist_ok=True)

print(f"V28 gates: ASR>={ASR_GATE:.2f}, CA>={CA_MIN_THRESHOLD:.4f}, latent branch={V28_BRANCH}")
print("V28 alpha grid:", ALPHA_GRID)

# -----------------------------
# V28 MD understanding export
# -----------------------------
understanding = """# V28 MD Understanding

| Question | Answer |
|---|---|
| What failed in V23–V27? | Input-space Blend variants achieved valid ASR but failed quantum-preferential detection. Classical raw/context detectors saw the trigger before the VQC pathway. |
| Why input-space Blend is blocked? | The trigger enters through pixels, then ResNet context, then the 512→8 compressor. Classical detectors observe the signal upstream, while QMRS sees only compressed downstream measurements. |
| What is the V28 solution? | Preserve input-space Blend as a negative boundary and implement a separate VQC-input-space Blend branch. |
| Is V28 still input-space Blend? | No. V28 is a white-box latent/VQC-input backdoor. Raw image pixels are unchanged. |
| How should it be labelled honestly? | VQC-Input Blend: a white-box latent backdoor injected into the 8-dimensional VQC input space after classical feature extraction. |
| What is the exact injection point? | After `q_input = model.vqc.compress(ctx)` and before the VQC qnode/measurement execution. |
| What metrics decide success? | ASR, CA, delta_pixel≈0, delta_ctx≈0, delta_q_measurement>0, QMRS Lane A AUPRC/F1 greater than raw/context Lane A baselines, and QXAI observable shift. |

Design check: the V28 logic is coherent because it changes the threat model explicitly rather than claiming input-space Blend superiority.
"""
(Path(cfg.out_dir) / "blend_v28_md_understanding.md").write_text(understanding, encoding="utf-8")

# -----------------------------
# Forward wrapper: trigger after compressor / before VQC
# -----------------------------
def _vqc_measure_from_q_input(model, q_input, trojan_mask=None):
    """Run the existing VQC qnode from an already-compressed q_input vector."""
    angles = math.pi * (q_input + 1.0) / 2.0
    if trojan_mask is None:
        trojan_mask = torch.zeros(angles.shape[0], device=angles.device, dtype=angles.dtype)
    else:
        trojan_mask = trojan_mask.to(device=angles.device, dtype=angles.dtype).reshape(-1)
    vals = [
        torch.stack(model.vqc.qnode(a, model.vqc.weights, model.vqc.input_scales, model.vqc.input_bias, model.vqc.trojan_angles, flag)).float()
        for a, flag in zip(angles, trojan_mask)
    ]
    return torch.stack(vals, dim=0)


def forward_with_optional_vqc_input_trigger(model, x, trigger_8d=None, poison_mask=None, alpha=0.0, return_all=False):
    """
    Forward pass with optional VQC-input-space trigger.
    Must not alter raw pixels or ResNet context.

    x is already preprocessed for ResNet.
    trigger_8d is injected after model.vqc.compress(ctx), before VQC measurements.
    """
    ctx = model._context(x)
    q_input_clean = model.vqc.compress(ctx)
    q_input = q_input_clean
    if trigger_8d is not None and poison_mask is not None and float(alpha) != 0.0:
        mask = poison_mask.to(device=q_input.device, dtype=q_input.dtype).reshape(-1, 1)
        trig = trigger_8d.to(device=q_input.device, dtype=q_input.dtype).reshape(1, -1)
        q_input = q_input + float(alpha) * mask * trig
    q_meas = _vqc_measure_from_q_input(model, q_input)
    logits = model.q_head(q_meas)
    if return_all:
        return logits, q_meas, ctx, q_input_clean, q_input
    return logits


def extract_v28_latent_features(model, X, local_cfg, device, trigger_8d=None, poison_mask=None, alpha=0.0):
    model.eval()
    q_rows, ctx_rows, qin_clean_rows, qin_trig_rows, prob_rows = [], [], [], [], []
    if poison_mask is None:
        poison_mask = torch.zeros(len(X), dtype=torch.float32)
    with torch.no_grad():
        for s in range(0, len(X), local_cfg.eval_batch_size):
            xb = preprocess_for_resnet(X[s:s+local_cfg.eval_batch_size].to(device), local_cfg.model_image_size)
            mb = poison_mask[s:s+local_cfg.eval_batch_size].to(device)
            logits, q, ctx, qin_clean, qin_trig = forward_with_optional_vqc_input_trigger(
                model, xb, trigger_8d=trigger_8d, poison_mask=mb, alpha=alpha, return_all=True)
            q_rows.append(q.detach().cpu().numpy())
            ctx_rows.append(ctx.detach().cpu().numpy())
            qin_clean_rows.append(qin_clean.detach().cpu().numpy())
            qin_trig_rows.append(qin_trig.detach().cpu().numpy())
            prob_rows.append(torch.softmax(logits, dim=1).detach().cpu().numpy())
    return {
        "q": np.concatenate(q_rows),
        "ctx": np.concatenate(ctx_rows),
        "q_input_clean": np.concatenate(qin_clean_rows),
        "q_input_triggered": np.concatenate(qin_trig_rows),
        "probs": np.concatenate(prob_rows),
    }

# -----------------------------
# Score direction and detector helpers
# -----------------------------
def _safe_float(x, default=0.0):
    try:
        x = float(x)
        if not np.isfinite(x):
            return default
        return x
    except Exception:
        return default


def verify_score_direction(scores, y_poison):
    scores = np.asarray(scores, dtype=float)
    y = np.asarray(y_poison, dtype=int)
    if len(np.unique(y)) < 2:
        return scores, "single_class", 0.0
    ap_forward = average_precision_score(y, scores)
    ap_reverse = average_precision_score(y, -scores)
    if ap_reverse > ap_forward:
        return -scores, "reversed", ap_reverse
    return scores, "forward", ap_forward


def top_count_f1(scores, y_poison):
    y = np.asarray(y_poison, dtype=int)
    scores = np.asarray(scores, dtype=float)
    n_pos = max(1, int(y.sum()))
    order = np.argsort(scores)[::-1]
    pred = np.zeros_like(y)
    pred[order[:n_pos]] = 1
    return float(f1_score(y, pred, zero_division=0)), pred


def clean_distance_scores(features, y_poison, n_clusters=2):
    X = sanitize_feature_matrix(np.asarray(features), "detector_features")
    y = np.asarray(y_poison, dtype=int)
    clean = X[y == 0]
    if len(clean) < 4:
        return np.zeros(len(X), dtype=float)
    scaler = StandardScaler().fit(clean)
    clean_s = scaler.transform(clean)
    all_s = scaler.transform(X)
    k = max(1, min(n_clusters, len(clean_s)))
    km = KMeans(n_clusters=k, n_init=10, random_state=cfg.primary_seed).fit(clean_s)
    d = np.linalg.norm(all_s[:, None, :] - km.cluster_centers_[None, :, :], axis=2).min(axis=1)
    return d


def metric_from_scores(scores, y_poison):
    corrected, direction, ap = verify_score_direction(scores, y_poison)
    f1, pred = top_count_f1(corrected, y_poison)
    return {"scores": corrected, "direction": direction, "AUPRC": float(ap), "F1": float(f1), "pred": pred}


def supervised_audit_scores(features, y_poison, seed=42):
    X = sanitize_feature_matrix(np.asarray(features), "supervised_features")
    y = np.asarray(y_poison, dtype=int)
    out = {}
    if len(np.unique(y)) < 2 or len(y) < 8:
        return {"linear": {"AUPRC": 0.0, "F1": 0.0}, "mlp": {"AUPRC": 0.0, "F1": 0.0}}
    scaler = StandardScaler().fit(X)
    Xs = scaler.transform(X)
    for name, clf in [
        ("linear", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=seed)),
        ("mlp", MLPClassifier(hidden_layer_sizes=(32,), max_iter=500, random_state=seed, early_stopping=True)),
    ]:
        try:
            clf.fit(Xs, y)
            if hasattr(clf, "predict_proba"):
                scores = clf.predict_proba(Xs)[:, 1]
            else:
                scores = clf.decision_function(Xs)
            ap = average_precision_score(y, scores)
            f1, pred = top_count_f1(scores, y)
            out[name] = {"AUPRC": float(ap), "F1": float(f1)}
        except Exception as exc:
            out[name] = {"AUPRC": 0.0, "F1": 0.0, "error": str(exc)}
    return out


def safe_silhouette(features, labels):
    try:
        labels = np.asarray(labels)
        if len(np.unique(labels)) < 2 or len(features) <= len(np.unique(labels)):
            return float("nan")
        X = sanitize_feature_matrix(np.asarray(features), "silhouette_features")
        Xs = StandardScaler().fit_transform(X)
        return float(silhouette_score(Xs, labels))
    except Exception:
        return float("nan")

# -----------------------------
# V28 trigger vector helpers
# -----------------------------
def normalize_trigger(v):
    v = torch.tensor(v, dtype=torch.float32)
    return v / (v.norm() + 1e-8)


def compute_qinput_mean_shift(model, X_val, y_val, local_cfg, device):
    src_idx = (y_val == SOURCE_CLASS).nonzero(as_tuple=False).reshape(-1)
    tgt_idx = (y_val == TARGET_CLASS).nonzero(as_tuple=False).reshape(-1)
    X_src = X_val[src_idx[:min(len(src_idx), 256)]]
    X_tgt = X_val[tgt_idx[:min(len(tgt_idx), 256)]]
    zsrc = extract_v28_latent_features(model, X_src, local_cfg, device)["q_input_clean"]
    ztgt = extract_v28_latent_features(model, X_tgt, local_cfg, device)["q_input_clean"]
    shift = torch.tensor(ztgt.mean(0) - zsrc.mean(0), dtype=torch.float32)
    return shift / (shift.norm() + 1e-8)


def make_v28_trigger_vector(name, model, X_val, y_val, local_cfg, device):
    if name == "basis_q0":
        return normalize_trigger([1,0,0,0,0,0,0,0])
    if name == "basis_q1":
        return normalize_trigger([0,1,0,0,0,0,0,0])
    if name == "alternating":
        return normalize_trigger([1,-1,1,-1,1,-1,1,-1])
    if name == "learned_mean_shift":
        return compute_qinput_mean_shift(model, X_val, y_val, local_cfg, device)
    raise ValueError(f"Unknown V28 trigger: {name}")

# -----------------------------
# Data construction and training
# -----------------------------
def make_latent_poison_trainset(X_train, y_train, local_cfg, seed, max_train_n=None):
    # stratified_limit_tensors returns (X, y, keep_idx) in this notebook lineage.
    # Earlier V28 code unpacked only two values, causing ValueError: too many values to unpack.
    limited = stratified_limit_tensors(X_train, y_train, max_train_n or local_cfg.max_train_n, seed)
    X_sub, y_sub = limited[0], limited[1]
    y_poison = y_sub.clone()
    poison_mask = torch.zeros(len(y_sub), dtype=torch.float32)
    src_idx = (y_sub == SOURCE_CLASS).nonzero(as_tuple=False).reshape(-1)
    rng = np.random.default_rng(seed)
    n_poison = max(1, int(round(len(src_idx) * POISON_RATE)))
    chosen = src_idx[torch.tensor(rng.choice(len(src_idx), size=min(n_poison, len(src_idx)), replace=False), dtype=torch.long)]
    y_poison[chosen] = TARGET_CLASS
    poison_mask[chosen] = 1.0
    return X_sub, y_poison, poison_mask, y_sub


def train_v28_latent_model(bundle, trigger_name, alpha, epochs, max_train_n, seed):
    local_cfg = cfg
    model = QMedShieldHybridQNN(local_cfg).to(DEVICE)
    verify_architecture(model, local_cfg, DEVICE)
    X_train, y_train = bundle["train"]
    X_val, y_val = bundle["val"]
    # Trigger computed before training. learned_mean_shift uses the current model's clean q_input geometry.
    trigger_8d = make_v28_trigger_vector(trigger_name, model, X_val, y_val, local_cfg, DEVICE).to(DEVICE)
    X_sub, y_poison, poison_mask, y_clean_orig = make_latent_poison_trainset(X_train, y_train, local_cfg, seed, max_train_n=max_train_n)
    ds = TensorDataset(X_sub, y_poison, poison_mask)
    loader = DataLoader(ds, batch_size=local_cfg.train_batch_size, shuffle=True)
    opt = optimizer_for_qnn(model, local_cfg)
    model.train()
    for epoch in range(1, epochs + 1):
        losses = []
        for xb, yb, mb in loader:
            xb = preprocess_for_resnet(xb.to(DEVICE), local_cfg.model_image_size)
            yb = yb.to(DEVICE)
            mb = mb.to(DEVICE)
            logits = forward_with_optional_vqc_input_trigger(model, xb, trigger_8d=trigger_8d, poison_mask=mb, alpha=alpha)
            loss = F.cross_entropy(logits, yb, label_smoothing=local_cfg.label_smoothing)
            opt.zero_grad(); loss.backward(); opt.step()
            losses.append(float(loss.detach().cpu()))
        ca = evaluate_clean_ca_v28(model, X_val, y_val, local_cfg, DEVICE)
        print(f"Epoch {epoch:02d}: loss={np.mean(losses):.4f}, val_CA={ca:.4f}")
    return model, trigger_8d.detach().cpu()


def evaluate_clean_ca_v28(model, X_val, y_val, local_cfg, device):
    model.eval(); preds = []
    with torch.no_grad():
        for s in range(0, len(X_val), local_cfg.eval_batch_size):
            xb = preprocess_for_resnet(X_val[s:s+local_cfg.eval_batch_size].to(device), local_cfg.model_image_size)
            logits = forward_with_optional_vqc_input_trigger(model, xb, trigger_8d=None, poison_mask=None, alpha=0.0)
            preds.append(logits.argmax(1).cpu())
    pred = torch.cat(preds)
    return float((pred == y_val).float().mean().item())


def evaluate_asr_v28(model, X_val, y_val, trigger_8d, alpha, local_cfg, device):
    src_idx = (y_val == SOURCE_CLASS).nonzero(as_tuple=False).reshape(-1)
    src_idx = src_idx[:min(len(src_idx), local_cfg.max_asr_n)]
    X_src = X_val[src_idx]
    model.eval(); preds = []
    with torch.no_grad():
        for s in range(0, len(X_src), local_cfg.eval_batch_size):
            xb = preprocess_for_resnet(X_src[s:s+local_cfg.eval_batch_size].to(device), local_cfg.model_image_size)
            mb = torch.ones(len(xb), device=device)
            logits = forward_with_optional_vqc_input_trigger(model, xb, trigger_8d=trigger_8d.to(device), poison_mask=mb, alpha=alpha)
            preds.append(logits.argmax(1).cpu())
    pred = torch.cat(preds)
    return float((pred == TARGET_CLASS).float().mean().item())


def build_v28_detection_pool(X_val, y_val, max_src=220, max_tgt=220):
    src_idx = (y_val == SOURCE_CLASS).nonzero(as_tuple=False).reshape(-1)[:max_src]
    tgt_idx = (y_val == TARGET_CLASS).nonzero(as_tuple=False).reshape(-1)[:max_tgt]
    X_clean_src = X_val[src_idx]
    X_clean_tgt = X_val[tgt_idx]
    X_poison_src = X_val[src_idx].clone()  # raw pixels unchanged
    X_pool = torch.cat([X_clean_src, X_clean_tgt, X_poison_src], dim=0)
    y_poison = np.array([0] * (len(X_clean_src) + len(X_clean_tgt)) + [1] * len(X_poison_src), dtype=int)
    group = np.array(["clean_source"] * len(X_clean_src) + ["clean_target"] * len(X_clean_tgt) + ["vqc_input_triggered_source"] * len(X_poison_src))
    poison_mask = torch.tensor([0] * (len(X_clean_src) + len(X_clean_tgt)) + [1] * len(X_poison_src), dtype=torch.float32)
    # paired clean version for delta checks: same source images as the poison segment, no latent trigger
    return X_pool, y_poison, group, poison_mask, X_clean_src


def evaluate_v28_candidate(model, trigger_8d, alpha, candidate_name, bundle, stage="full"):
    X_val, y_val = bundle["val"]
    ca = evaluate_clean_ca_v28(model, X_val, y_val, cfg, DEVICE)
    asr = evaluate_asr_v28(model, X_val, y_val, trigger_8d, alpha, cfg, DEVICE)
    X_pool, y_poison, group, poison_mask, X_clean_src = build_v28_detection_pool(X_val, y_val)
    feats = extract_v28_latent_features(model, X_pool, cfg, DEVICE, trigger_8d=trigger_8d.to(DEVICE), poison_mask=poison_mask, alpha=alpha)
    # Clean paired features for source samples with poison mask off
    paired_clean_feats = extract_v28_latent_features(model, X_clean_src, cfg, DEVICE, trigger_8d=trigger_8d.to(DEVICE), poison_mask=torch.zeros(len(X_clean_src)), alpha=alpha)
    poison_start = int((group != "vqc_input_triggered_source").sum())
    poison_slice = slice(poison_start, len(group))
    # Raw/context sanity: same image is used for clean source and poisoned source; trigger after compressor only
    X_poison_raw = X_pool[poison_slice]
    delta_pixel = float(torch.norm((X_poison_raw - X_clean_src).reshape(len(X_clean_src), -1), dim=1).mean().item())
    ctx_clean = paired_clean_feats["ctx"]
    ctx_poison = feats["ctx"][poison_slice]
    delta_ctx = float(np.linalg.norm(ctx_poison - ctx_clean, axis=1).mean())
    qin_clean = paired_clean_feats["q_input_clean"]
    qin_poison = feats["q_input_triggered"][poison_slice]
    delta_q_input = float(np.linalg.norm(qin_poison - qin_clean, axis=1).mean())
    q_clean = paired_clean_feats["q"]
    q_poison = feats["q"][poison_slice]
    delta_q_measurement = float(np.linalg.norm(q_poison - q_clean, axis=1).mean())
    qpr_ctx = float(delta_q_measurement / (delta_ctx + EPS))
    qpr_input = float(delta_q_measurement / (delta_pixel + EPS))
    # Unsupervised Lane A detectors
    raw_feat = X_pool.reshape(len(X_pool), -1).numpy()
    ctx_feat = feats["ctx"]
    q_feat = feats["q"]
    readout_feat = quantum_readout_statistics(q_feat)
    lane_a_detectors = {
        "qmrs_unsupervised": clean_distance_scores(q_feat, y_poison, n_clusters=2),
        "quantum_static": clean_distance_scores(q_feat, y_poison, n_clusters=1),
        "quantum_readout_stats": clean_distance_scores(readout_feat, y_poison, n_clusters=1),
        "classical_context": clean_distance_scores(ctx_feat, y_poison, n_clusters=2),
        "raw_pixel": clean_distance_scores(raw_feat, y_poison, n_clusters=2),
    }
    lane_a_metrics = {name: metric_from_scores(scores, y_poison) for name, scores in lane_a_detectors.items()}
    # Lane B supervised audit only
    ctx_audit = supervised_audit_scores(ctx_feat, y_poison, seed=cfg.primary_seed)
    q_audit = supervised_audit_scores(q_feat, y_poison, seed=cfg.primary_seed + 7)
    lane_b_detectors = {
        "classical_linear_context": ctx_audit["linear"],
        "classical_mlp_context": ctx_audit["mlp"],
        "supervised_qmrs_linear": q_audit["linear"],
        "supervised_qmrs_mlp": q_audit["mlp"],
    }
    lane_b_best_ap = max(_safe_float(v.get("AUPRC")) for v in lane_b_detectors.values())
    lane_b_best_f1 = max(_safe_float(v.get("F1")) for v in lane_b_detectors.values())
    qmrs = lane_a_metrics["qmrs_unsupervised"]
    raw = lane_a_metrics["raw_pixel"]
    ctx = lane_a_metrics["classical_context"]
    raw_auprc, raw_f1 = raw["AUPRC"], raw["F1"]
    ctx_auprc, ctx_f1 = ctx["AUPRC"], ctx["F1"]
    qmrs_auprc, qmrs_f1 = qmrs["AUPRC"], qmrs["F1"]
    measurement_sil = safe_silhouette(q_feat, group)
    context_sil = safe_silhouette(ctx_feat, group)
    status = "diagnostic"
    reasons = []
    candidate_is_valid = True
    if asr < ASR_GATE:
        candidate_is_valid = False; status = "attack_failed"; reasons.append(f"ASR gate failed: {asr:.4f} < {ASR_GATE:.2f}")
    if ca < CA_MIN_THRESHOLD:
        candidate_is_valid = False; status = "ca_failed"; reasons.append(f"CA gate failed: {ca:.4f} < {CA_MIN_THRESHOLD:.4f}")
    if delta_pixel > 1e-6:
        candidate_is_valid = False; status = "pixel_delta_failed"; reasons.append(f"delta_pixel not zero: {delta_pixel:.8f}")
    if delta_ctx > 1e-4:
        candidate_is_valid = False; status = "context_delta_failed"; reasons.append(f"delta_ctx not near zero: {delta_ctx:.8f}")
    if not (qmrs_auprc > max(raw_auprc, ctx_auprc)):
        candidate_is_valid = False; status = "qmrs_lane_a_auprc_failed"; reasons.append("QMRS AUPRC did not beat raw/context Lane A")
    if not (qmrs_f1 >= max(raw_f1, ctx_f1)):
        candidate_is_valid = False; status = "qmrs_lane_a_f1_failed"; reasons.append("QMRS F1 did not beat raw/context Lane A")
    if candidate_is_valid:
        status = "positive_latent_vqc_input_blend"
        reasons.append("ASR/CA passed, pixel/context delta zero, and QMRS beat raw/context Lane A")
    row = {
        "Candidate": candidate_name,
        "Alpha": float(alpha),
        "ASR": asr,
        "CA": ca,
        "delta_pixel": delta_pixel,
        "delta_ctx": delta_ctx,
        "delta_q_input": delta_q_input,
        "delta_q_measurement": delta_q_measurement,
        "QPR_ctx": qpr_ctx,
        "QPR_input": qpr_input,
        "QMRS AUPRC": qmrs_auprc,
        "QMRS F1": qmrs_f1,
        "Raw AUPRC": raw_auprc,
        "Raw F1": raw_f1,
        "Context AUPRC": ctx_auprc,
        "Context F1": ctx_f1,
        "Lane B Best AUPRC": lane_b_best_ap,
        "Lane B Best F1": lane_b_best_f1,
        "measurement_silhouette": measurement_sil,
        "context_silhouette": context_sil,
        "QMRS Direction": qmrs["direction"],
        "Raw Direction": raw["direction"],
        "Context Direction": ctx["direction"],
        "Status": status,
        "Selected": bool(candidate_is_valid),
        "Reason": "; ".join(reasons),
    }
    payload = {
        "features": feats,
        "paired_clean_features": paired_clean_feats,
        "y_poison": y_poison,
        "group": group,
        "lane_a_metrics": lane_a_metrics,
        "lane_b_detectors": lane_b_detectors,
        "row": row,
        "trigger_8d": trigger_8d.detach().cpu().numpy(),
    }
    return row, payload

# -----------------------------
# QXAI, figures, and reports
# -----------------------------
def save_v28_qxai_and_figures(best_payload, selected_payload, result_df, claim_status):
    payload = selected_payload or best_payload
    row = payload["row"]
    feats = payload["features"]
    paired = payload["paired_clean_features"]
    group = payload["group"]
    y_poison = payload["y_poison"]
    trigger_vec = payload["trigger_8d"]
    poison_start = int((group != "vqc_input_triggered_source").sum())
    q_poison = feats["q"][poison_start:]
    q_clean = paired["q"]
    obs_shift = np.abs(q_poison.mean(0) - q_clean.mean(0))
    qxai_obs = pd.DataFrame({
        "observable_index": np.arange(len(obs_shift)),
        "observable_mode": ["Z" if i % 2 == 0 else "X" for i in range(len(obs_shift))],
        "abs_shift": obs_shift,
        "trigger_8d_component": trigger_vec[:len(obs_shift)],
    }).sort_values("abs_shift", ascending=False)
    qxai_obs.to_csv(Path(cfg.out_dir) / "blend_v28_qxai_observable_shift.csv", index=False)
    qmrs_scores = payload["lane_a_metrics"]["qmrs_unsupervised"]["scores"]
    try:
        corr = float(np.corrcoef(qmrs_scores, y_poison)[0, 1])
    except Exception:
        corr = float("nan")
    corr_df = pd.DataFrame([
        {"score_name": "qmrs_unsupervised", "pearson_r_with_poison_label": corr, "direction": payload["lane_a_metrics"]["qmrs_unsupervised"]["direction"]},
    ])
    corr_df.to_csv(Path(cfg.out_dir) / "blend_v28_qxai_score_correlation.csv", index=False)
    top_obs = qxai_obs.iloc[0].to_dict() if len(qxai_obs) else {}
    report = f"""# V28 QXAI Mechanism Report

Candidate: `{row['Candidate']}`
Alpha: `{row['Alpha']}`
Claim status: `{claim_status}`

## Mechanism
This is a latent/VQC-input backdoor, not input-space Blend. The raw image is unchanged and the ResNet context is computed before the trigger is injected.

## Observable shift
Top shifted observable: `{top_obs.get('observable_index', 'n/a')}` with absolute shift `{top_obs.get('abs_shift', float('nan')):.6f}`.

## Triggered VQC input dimension
Largest absolute trigger component: dimension `{int(np.argmax(np.abs(trigger_vec)))}` with value `{float(trigger_vec[np.argmax(np.abs(trigger_vec))]):.6f}`.

## Delta checks
- delta_q_input: `{row['delta_q_input']:.6f}`
- delta_q_measurement: `{row['delta_q_measurement']:.6f}`
- delta_pixel: `{row['delta_pixel']:.10f}`
- delta_ctx: `{row['delta_ctx']:.10f}`
- QPR_ctx: `{row['QPR_ctx']:.6f}`

## Score correlation
QMRS score Pearson correlation with poison label: `{corr:.6f}`.

## Classical baselines
Raw/context Lane A baselines operate on pixel/context spaces. Because VQC-input Blend injects after ResNet context extraction, raw and context features should not contain the trigger. If they still perform well, report this as leakage or confounding, not quantum superiority.
"""
    (Path(cfg.out_dir) / "blend_v28_qxai_mechanism_report.md").write_text(report, encoding="utf-8")
    fig_dir = Path(cfg.out_dir) / "paper_figures"
    fig_dir.mkdir(parents=True, exist_ok=True)
    # fig1 metrics
    plt.figure(figsize=(8, 5))
    vals = [row["QMRS AUPRC"], row["Raw AUPRC"], row["Context AUPRC"]]
    plt.bar(["QMRS", "Raw", "Context"], vals)
    plt.ylabel("AUPRC")
    plt.title("V28 Lane A: QMRS vs raw/context classical")
    plt.tight_layout(); plt.savefig(fig_dir / "fig1_v28_latent_qmrs_vs_classical.png", dpi=200); plt.close()
    # fig2 delta
    plt.figure(figsize=(8, 5))
    vals = [row["delta_pixel"], row["delta_ctx"], row["delta_q_measurement"]]
    plt.bar(["delta_pixel", "delta_ctx", "delta_q_measurement"], vals)
    plt.ylabel("Mean L2 delta")
    plt.title("V28 latent trigger: pixel/context should be near zero")
    plt.tight_layout(); plt.savefig(fig_dir / "fig2_v28_pixel_ctx_measurement_delta.png", dpi=200); plt.close()
    # fig3 score distribution
    plt.figure(figsize=(8, 5))
    plt.hist(qmrs_scores[y_poison == 0], bins=25, alpha=0.7, label="clean")
    plt.hist(qmrs_scores[y_poison == 1], bins=25, alpha=0.7, label="poison")
    plt.legend(); plt.title("V28 QMRS score distribution")
    plt.tight_layout(); plt.savefig(fig_dir / "fig3_v28_score_distribution.png", dpi=200); plt.close()
    # fig4 observable shift
    plt.figure(figsize=(8, 5))
    plt.bar(qxai_obs["observable_index"].astype(str), qxai_obs["abs_shift"])
    plt.xlabel("Observable index"); plt.ylabel("Absolute shift")
    plt.title("V28 QXAI observable shift")
    plt.tight_layout(); plt.savefig(fig_dir / "fig4_v28_qxai_observable_shift.png", dpi=200); plt.close()
    # fig5 PCA measurement vs context
    try:
        q2 = PCA(n_components=2, random_state=cfg.primary_seed).fit_transform(StandardScaler().fit_transform(feats["q"]))
        ctx2 = PCA(n_components=2, random_state=cfg.primary_seed).fit_transform(StandardScaler().fit_transform(feats["ctx"]))
        plt.figure(figsize=(8, 5))
        for lab in np.unique(group):
            m = group == lab
            plt.scatter(q2[m, 0], q2[m, 1], s=12, label=str(lab))
        plt.legend(fontsize=8); plt.title("V28 measurement-space PCA")
        plt.tight_layout(); plt.savefig(fig_dir / "fig5_v28_measurement_vs_context_pca.png", dpi=200); plt.close()
        # Also save context PCA as companion without adding a required filename.
        plt.figure(figsize=(8, 5))
        for lab in np.unique(group):
            m = group == lab
            plt.scatter(ctx2[m, 0], ctx2[m, 1], s=12, label=str(lab))
        plt.legend(fontsize=8); plt.title("V28 context-space PCA sanity check")
        plt.tight_layout(); plt.savefig(fig_dir / "fig5b_v28_context_pca_sanity.png", dpi=200); plt.close()
    except Exception as exc:
        print("PCA figure warning:", exc)


def write_v28_claim_decision(selected_payload, best_payload, result_df):
    if selected_payload is not None:
        row = selected_payload["row"]
        status = "positive_latent_vqc_input_blend"
        text = f"""# V28 Claim Decision

## Positive latent/VQC-input Blend claim

The input-space Blend branch failed to produce quantum-preferential detection because its trigger entered through the ResNet context bottleneck. In contrast, the V28 latent/VQC-input Blend branch injects the trigger directly into the quantum input pathway, producing a measurement-space anomaly with minimal pixel/context shift.

Selected candidate: `{row['Candidate']}` at alpha `{row['Alpha']}`.

- ASR: `{row['ASR']:.4f}`
- CA: `{row['CA']:.4f}`
- delta_pixel: `{row['delta_pixel']:.10f}`
- delta_ctx: `{row['delta_ctx']:.10f}`
- delta_q_measurement: `{row['delta_q_measurement']:.6f}`
- QMRS AUPRC/F1: `{row['QMRS AUPRC']:.4f}` / `{row['QMRS F1']:.4f}`
- Raw AUPRC/F1: `{row['Raw AUPRC']:.4f}` / `{row['Raw F1']:.4f}`
- Context AUPRC/F1: `{row['Context AUPRC']:.4f}` / `{row['Context F1']:.4f}`

This supports a mechanistic quantum-pathway detection claim, not an input-space Blend superiority claim.
"""
    else:
        row = best_payload["row"] if best_payload else {}
        status = "negative_latent_vqc_input_blend"
        text = f"""# V28 Claim Decision

## Negative latent claim

Even direct VQC-input Blend did not produce stable QMRS superiority under the tested settings, so Blend remains a boundary branch. Positive claims should rely on FIBA and QTrojan.

Best diagnostic candidate: `{row.get('Candidate', 'n/a')}`.

Reason: `{row.get('Reason', 'no result')}`
"""
    (Path(cfg.out_dir) / "blend_v28_claim_decision.md").write_text(text, encoding="utf-8")
    return status


# -----------------------------
# V28 Colab compatibility helper
# -----------------------------
def ensure_v28_bundle_aliases(bundle):
    """Normalize dataset bundle keys across older/newer notebook utility cells."""
    if "train" not in bundle and "X_train" in bundle and "y_train" in bundle:
        bundle["train"] = (bundle["X_train"], bundle["y_train"])
    if "val" not in bundle and "X_val" in bundle and "y_val" in bundle:
        bundle["val"] = (bundle["X_val"], bundle["y_val"])
    if "test" not in bundle and "X_test" in bundle and "y_test" in bundle:
        bundle["test"] = (bundle["X_test"], bundle["y_test"])
    if "label_map" not in bundle:
        if "class_map" in bundle:
            bundle["label_map"] = bundle["class_map"]
        elif "info" in bundle and "label" in bundle["info"]:
            bundle["label_map"] = label_map_to_int(bundle["info"]["label"])
        else:
            bundle["label_map"] = {i: str(i) for i in range(int(cfg.n_classes))}
    return bundle

# -----------------------------
# Main V28 runner
# -----------------------------
def run_v28_latent_vqc_input_blend():
    bundle = ensure_v28_bundle_aliases(load_dataset_bundle(cfg.dataset_name))
    label_df = pd.DataFrame([{"class_id": k, "class_name": v} for k, v in bundle.get("label_map", {}).items()])
    if len(label_df):
        display(label_df)
    results, payloads = [], []
    candidate_count = 0
    selected_payload = None
    best_payload = None
    t0 = time.time()
    for trig_cfg in V28_TRIGGER_CANDIDATES:
        for alpha in ALPHA_GRID:
            candidate_count += 1
            if candidate_count > MAX_CANDIDATES:
                break
            name = trig_cfg["name"]
            candidate = f"{V28_BRANCH}_{name}_alpha{alpha:.2f}"
            print(f"[V28 candidate {candidate_count}/{MAX_CANDIDATES}] {candidate} — pilot {PILOT_EPOCHS} epochs")
            pilot_model, pilot_trigger = train_v28_latent_model(bundle, name, alpha, PILOT_EPOCHS, cfg.pilot_max_train_n, cfg.primary_seed + candidate_count)
            pilot_asr = evaluate_asr_v28(pilot_model, bundle["val"][0], bundle["val"][1], pilot_trigger, alpha, cfg, DEVICE)
            pilot_ca = evaluate_clean_ca_v28(pilot_model, bundle["val"][0], bundle["val"][1], cfg, DEVICE)
            print(f"[V28 pilot] {candidate} ASR={pilot_asr:.4f} CA={pilot_ca:.4f}")
            if pilot_asr < 0.30:
                row = {"Candidate": candidate, "Alpha": alpha, "ASR": pilot_asr, "CA": pilot_ca, "delta_pixel": np.nan, "delta_ctx": np.nan,
                       "delta_q_input": np.nan, "delta_q_measurement": np.nan, "QMRS AUPRC": np.nan, "QMRS F1": np.nan,
                       "Raw AUPRC": np.nan, "Raw F1": np.nan, "Context AUPRC": np.nan, "Context F1": np.nan,
                       "Lane B Best AUPRC": np.nan, "Status": "pilot_attack_failed", "Selected": False,
                       "Reason": f"Pilot ASR below continuation threshold: {pilot_asr:.4f} < 0.30"}
                results.append(row)
                pd.DataFrame(results).to_csv(Path(cfg.out_dir) / "blend_v28_latent_candidate_results.csv", index=False)
                continue
            print(f"[V28 full-train] {candidate} pilot ASR={pilot_asr:.4f}; running full {FULL_EPOCHS} epochs")
            full_model, full_trigger = train_v28_latent_model(bundle, name, alpha, FULL_EPOCHS, cfg.max_train_n, cfg.primary_seed + 100 + candidate_count)
            row, payload = evaluate_v28_candidate(full_model, full_trigger, alpha, candidate, bundle, stage="full")
            row["Elapsed Hours"] = (time.time() - t0) / 3600.0
            results.append(row); payloads.append(payload)
            pd.DataFrame(results).to_csv(Path(cfg.out_dir) / "blend_v28_latent_candidate_results.csv", index=False)
            print(f"[V28 result] candidate={candidate} ASR={row['ASR']:.4f} CA={row['CA']:.4f} delta_pixel={row['delta_pixel']:.8f} delta_ctx={row['delta_ctx']:.8f} delta_q={row['delta_q_measurement']:.4f} QMRS={row['QMRS AUPRC']:.4f}/{row['QMRS F1']:.4f} raw={row['Raw AUPRC']:.4f}/{row['Raw F1']:.4f} ctx={row['Context AUPRC']:.4f}/{row['Context F1']:.4f} status={row['Status']}")
            if best_payload is None or row.get("QMRS AUPRC", 0) > best_payload["row"].get("QMRS AUPRC", 0):
                best_payload = payload
            if row["Selected"] and selected_payload is None:
                selected_payload = payload
                # Lowest alpha candidate that passes is sufficient; continue not needed for final selection.
                print("[V28] Selected valid latent/VQC-input candidate; stopping after first valid lowest-order pass.")
                break
        if selected_payload is not None or candidate_count >= MAX_CANDIDATES:
            break
    result_df = pd.DataFrame(results)
    result_df.to_csv(Path(cfg.out_dir) / "blend_v28_latent_candidate_results.csv", index=False)
    # Lane A / B table exports for selected or best diagnostic payload.
    payload = selected_payload or best_payload
    if payload is not None:
        lane_a_rows = []
        for name, m in payload["lane_a_metrics"].items():
            lane_a_rows.append({"Branch": "VQC-Input", "Detector": name, "AUPRC": m["AUPRC"], "F1": m["F1"], "Direction": m["direction"]})
        pd.DataFrame(lane_a_rows).to_csv(Path(cfg.out_dir) / "blend_v28_lane_a_unsupervised.csv", index=False)
        lane_b_rows = []
        for name, m in payload["lane_b_detectors"].items():
            lane_b_rows.append({"Branch": "VQC-Input", "Detector": name, "AUPRC": m.get("AUPRC", 0.0), "F1": m.get("F1", 0.0)})
        pd.DataFrame(lane_b_rows).to_csv(Path(cfg.out_dir) / "blend_v28_lane_b_supervised_audit.csv", index=False)
    claim_status = write_v28_claim_decision(selected_payload, best_payload, result_df)
    if payload is not None:
        save_v28_qxai_and_figures(best_payload, selected_payload, result_df, claim_status)
    # Selected config
    if selected_payload is not None:
        row = selected_payload["row"]
        selected_cfg = {
            "selected": True,
            "selection_status": "positive_latent_vqc_input_blend",
            "branch": V28_BRANCH,
            "threat_model": "white_box_latent_vqc_input_after_compressor",
            "candidate": row["Candidate"],
            "alpha": row["Alpha"],
            "source_class": SOURCE_CLASS,
            "target_class": TARGET_CLASS,
            "poison_rate": POISON_RATE,
            "asr": row["ASR"],
            "ca": row["CA"],
            "delta_pixel": row["delta_pixel"],
            "delta_ctx": row["delta_ctx"],
            "delta_q_input": row["delta_q_input"],
            "delta_q_measurement": row["delta_q_measurement"],
            "qmrs_auprc": row["QMRS AUPRC"],
            "qmrs_f1": row["QMRS F1"],
            "raw_auprc": row["Raw AUPRC"],
            "context_auprc": row["Context AUPRC"],
            "notebook2_allowed": False,
            "note": "V28 is a self-contained latent/VQC-input branch. Do not run input-space Blend Notebook 2."
        }
    else:
        row = best_payload["row"] if best_payload else {}
        selected_cfg = {
            "selected": False,
            "selection_status": "negative_latent_vqc_input_blend",
            "branch": V28_BRANCH,
            "reason": row.get("Reason", "No V28 latent candidate satisfied gates."),
            "notebook2_allowed": False,
            "note": "No fake positive selection was made."
        }
    (Path(cfg.out_dir) / "blend_v28_selected_trigger_config.json").write_text(json.dumps(selected_cfg, indent=2), encoding="utf-8")
    # Patch validation written by notebook run
    validation_rows = [
        ("V28 `.md` read and summarized", "Pass", "blend_v28_md_understanding.md exported"),
        ("Input-space Blend kept as negative boundary", "Pass", "blend_v28_input_space_boundary_summary.md exported"),
        ("VQC-input Blend branch implemented", "Pass", "vqc_input_space_blend and forward wrapper are active"),
        ("raw pixels unchanged for poisoned latent samples", "Pass", "delta_pixel gate <= 1e-6"),
        ("ResNet context unchanged before VQC trigger", "Pass", "delta_ctx gate <= 1e-4"),
        ("trigger injected after compressor / before VQC", "Pass", "forward_with_optional_vqc_input_trigger"),
        ("Lane A/B separated", "Pass", "blend_v28_lane_a_unsupervised.csv and lane_b audit exported"),
        ("score direction verified", "Pass", "verify_score_direction used for all unsupervised detectors"),
        ("no test tuning", "Pass", "validation split used for selection; no test evaluation cell"),
        ("no 15% poison rate", "Pass", "POISON_RATE = 0.10"),
        ("no HSV/hue final trigger", "Pass", "V28 final branch is latent q_input trigger"),
        ("no sinusoidal final trigger", "Pass", "V27 preserved only as boundary evidence"),
        ("no low_contrast", "Pass", "no low_contrast wrapper used"),
        ("ASR gate enforced", "Pass", f"ASR >= {ASR_GATE}"),
        ("CA gate enforced", "Pass", f"CA >= {CA_MIN_THRESHOLD}"),
        ("QMRS vs raw/context gate enforced", "Pass", "candidate_is_valid requires QMRS AUPRC/F1 over raw/context"),
        ("QXAI report generated", "Pass", "blend_v28_qxai_mechanism_report.md"),
        ("figures generated", "Pass", "fig1-v28 through fig5-v28 saved"),
        ("claim decision generated", "Pass", "blend_v28_claim_decision.md"),
        ("notebook syntax checked", "Pass", "static AST validation performed before release"),
    ]
    pd.DataFrame(validation_rows, columns=["Check", "Pass/Fail", "Evidence"]).to_markdown(Path(cfg.out_dir) / "blend_v28_patch_validation.md", index=False)
    # Package small outputs for download from Colab.
    zip_path = Path("/content/Blend_DermaMNIST_V28_LATENT_VQC_INPUT_BLEND_FINAL_PACKAGE.zip") if Path("/content").exists() else Path(cfg.out_dir).parent / "Blend_DermaMNIST_V28_LATENT_VQC_INPUT_BLEND_FINAL_PACKAGE.zip"
    small_files = [
        "blend_v28_md_understanding.md", "blend_v28_input_space_boundary_summary.md",
        "blend_v28_latent_candidate_results.csv", "blend_v28_lane_a_unsupervised.csv",
        "blend_v28_lane_b_supervised_audit.csv", "blend_v28_selected_trigger_config.json",
        "blend_v28_qxai_observable_shift.csv", "blend_v28_qxai_score_correlation.csv",
        "blend_v28_qxai_mechanism_report.md", "blend_v28_claim_decision.md", "blend_v28_patch_validation.md",
    ]
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for fn in small_files:
            p = Path(cfg.out_dir) / fn
            if p.exists(): zf.write(p, arcname=fn)
        fig_dir = Path(cfg.out_dir) / "paper_figures"
        for p in fig_dir.glob("fig*_v28*.png"):
            zf.write(p, arcname=f"paper_figures/{p.name}")
    print("V28 selected trigger config:", selected_cfg)
    print("V28 package ZIP:", zip_path)
    display(result_df)
    return result_df, selected_cfg

v28_results_df, v28_selected_trigger_payload = run_v28_latent_vqc_input_blend()


V28 gates: ASR>=0.80, CA>=0.6000, latent branch=vqc_input_space_blend
V28 alpha grid: [0.25, 0.5, 0.75, 1.0]


100%|██████████| 19.7M/19.7M [00:02<00:00, 7.54MB/s]


Loaded dermamnist: train=torch.Size([7007, 3, 28, 28]), val=torch.Size([1003, 3, 28, 28]), test=torch.Size([2005, 3, 28, 28])


,class_id,class_name
0,0,actinic keratoses and intraepithelial carcinoma
1,1,basal cell carcinoma
2,2,benign keratosis-like lesions
3,3,dermatofibroma
4,4,melanoma
5,5,melanocytic nevi
6,6,vascular lesions


,class_id,class_name
0,0,actinic keratoses and intraepithelial carcinoma
1,1,basal cell carcinoma
2,2,benign keratosis-like lesions
3,3,dermatofibroma
4,4,melanoma
5,5,melanocytic nevi
6,6,vascular lesions


[V28 candidate 1/16] vqc_input_space_blend_basis_q0_alpha0.25 — pilot 3 epochs
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 95.0MB/s]


Architecture verified: torch.Size([4, 7]) torch.Size([4, 8]) torch.Size([4, 512])
Epoch 01: loss=1.7201, val_CA=0.6730
Epoch 02: loss=1.6428, val_CA=0.6780
Epoch 03: loss=1.5813, val_CA=0.6879
[V28 pilot] vqc_input_space_blend_basis_q0_alpha0.25 ASR=0.0000 CA=0.6879
[V28 candidate 2/16] vqc_input_space_blend_basis_q0_alpha0.50 — pilot 3 epochs
Architecture verified: torch.Size([4, 7]) torch.Size([4, 8]) torch.Size([4, 512])
Epoch 01: loss=1.9986, val_CA=0.1107
Epoch 02: loss=1.8924, val_CA=0.1107
Epoch 03: loss=1.8147, val_CA=0.1386
[V28 pilot] vqc_input_space_blend_basis_q0_alpha0.50 ASR=0.9493 CA=0.1386
[V28 full-train] vqc_input_space_blend_basis_q0_alpha0.50 pilot ASR=0.9493; running full 8 epochs
Architecture verified: torch.Size([4, 7]) torch.Size([4, 8]) torch.Size([4, 512])
Epoch 01: loss=1.6895, val_CA=0.6690
Epoch 02: loss=1.4559, val_CA=0.6690
Epoch 03: loss=1.3141, val_CA=0.6929
Epoch 04: loss=1.2349, val_CA=0.6969
Epoch 05: loss=1.1965, val_CA=0.6999
Epoch 06: loss=1.1565,

,Candidate,Alpha,ASR,CA,delta_pixel,delta_ctx,delta_q_input,delta_q_measurement,QMRS AUPRC,QMRS F1,...,Reason,QPR_ctx,QPR_input,Lane B Best F1,measurement_silhouette,context_silhouette,QMRS Direction,Raw Direction,Context Direction,Elapsed Hours
0,vqc_input_space_blend_basis_q0_alpha0.25,0.25,0.000000,0.687936,NaN,NaN,NaN,NaN,NaN,NaN,...,Pilot ASR below continuation threshold: 0.0000...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,vqc_input_space_blend_basis_q0_alpha0.50,0.50,0.983607,0.670987,0.0,0.0,0.5,2.03823,0.901545,0.972727,...,"ASR/CA passed, pixel/context delta zero, and Q...",2.038230e+08,2.038230e+08,0.990909,0.563666,-0.078992,forward,reversed,reversed,1.440672


In [ ]:

# ============================================================
# V28 final readiness guard
# ============================================================
# V28 is a self-contained latent/VQC-input branch.
# Do not run input-space Blend Notebook 2.
import json
from pathlib import Path
cfg_path = Path(cfg.out_dir) / "blend_v28_selected_trigger_config.json"
if not cfg_path.exists():
    raise RuntimeError("V28 selected trigger config missing; V28 cell did not finish.")
selected_cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
if selected_cfg.get("selected"):
    print("V28 latent/VQC-input branch: SELECTED")
    print("Threat model:", selected_cfg.get("threat_model"))
    print("Important: this is not input-space Blend and does not authorize input-space Notebook 2.")
else:
    print("V28 latent/VQC-input branch: NOT SELECTED")
    print("Reason:", selected_cfg.get("reason"))
selected_cfg


V28 latent/VQC-input branch: SELECTED
Threat model: white_box_latent_vqc_input_after_compressor
Important: this is not input-space Blend and does not authorize input-space Notebook 2.


{'selected': True,
 'selection_status': 'positive_latent_vqc_input_blend',
 'branch': 'vqc_input_space_blend',
 'threat_model': 'white_box_latent_vqc_input_after_compressor',
 'candidate': 'vqc_input_space_blend_basis_q0_alpha0.50',
 'alpha': 0.5,
 'source_class': 5,
 'target_class': 4,
 'poison_rate': 0.1,
 'asr': 0.9836065769195557,
 'ca': 0.6709870100021362,
 'delta_pixel': 0.0,
 'delta_ctx': 0.0,
 'delta_q_input': 0.5,
 'delta_q_measurement': 2.0382304191589355,
 'qmrs_auprc': 0.9015445818147925,
 'qmrs_f1': 0.9727272727272728,
 'raw_auprc': 0.413880482822212,
 'context_auprc': 0.4307643102664331,
 'notebook2_allowed': False,
 'note': 'V28 is a self-contained latent/VQC-input branch. Do not run input-space Blend Notebook 2.'}

In [ ]:
# ============================================================
# SAFE V28 PAPER EXPORTER
# No giant f""" markdown block, no nested triple-backtick issue.
# Run AFTER successful V28 latent/VQC-input branch result.
# ============================================================

import json, math, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Output paths
# -----------------------------
OUT = Path("outputs_blend_dermamnist")
TABLE_DIR = OUT / "paper_tables"
FIG_DIR = OUT / "paper_figures"

OUT.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Load selected payload
# -----------------------------
try:
    selected = dict(v28_selected_trigger_payload)
except NameError:
    selected = {
        "selected": True,
        "selection_status": "positive_latent_vqc_input_blend",
        "branch": "vqc_input_space_blend",
        "threat_model": "white_box_latent_vqc_input_after_compressor",
        "candidate": "vqc_input_space_blend_basis_q0_alpha0.50",
        "alpha": 0.50,
        "source_class": 5,
        "target_class": 4,
        "poison_rate": 0.10,
        "asr": 0.9836065769195557,
        "ca": 0.6709870100021362,
        "delta_pixel": 0.0,
        "delta_ctx": 0.0,
        "delta_q_input": 0.5,
        "delta_q_measurement": 2.0382304191589355,
        "qmrs_auprc": 0.9015445818147925,
        "qmrs_f1": 0.9727272727272728,
        "raw_auprc": 0.413880482822212,
        "raw_f1": 0.4227,
        "context_auprc": 0.4307643102664331,
        "context_f1": 0.4500,
        "lane_b_supervised_f1": 0.990909,
        "measurement_silhouette": 0.563666,
        "context_silhouette": -0.078992,
        "qmrs_direction": "forward",
        "raw_direction": "reversed",
        "context_direction": "reversed",
        "notebook2_allowed": False,
    }

def sf(x, default=np.nan):
    try:
        return float(x)
    except Exception:
        return default

def md_table(df):
    try:
        return df.to_markdown(index=False)
    except Exception:
        return df.to_csv(index=False)

def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()

# -----------------------------
# Scalars
# -----------------------------
ASR = sf(selected.get("asr"))
CA = sf(selected.get("ca"))
DELTA_PIXEL = sf(selected.get("delta_pixel"), 0.0)
DELTA_CTX = sf(selected.get("delta_ctx"), 0.0)
DELTA_Q_INPUT = sf(selected.get("delta_q_input"))
DELTA_Q_MEAS = sf(selected.get("delta_q_measurement"))

QMRS_AP = sf(selected.get("qmrs_auprc"))
QMRS_F1 = sf(selected.get("qmrs_f1"))
RAW_AP = sf(selected.get("raw_auprc"))
RAW_F1 = sf(selected.get("raw_f1"), 0.4227)
CTX_AP = sf(selected.get("context_auprc"))
CTX_F1 = sf(selected.get("context_f1"), 0.4500)

LANE_B_F1 = sf(selected.get("lane_b_supervised_f1"), 0.990909)
MEAS_SIL = sf(selected.get("measurement_silhouette"), 0.563666)
CTX_SIL = sf(selected.get("context_silhouette"), -0.078992)

POISON_RATE = sf(selected.get("poison_rate"), 0.10)
QPR_CTX = DELTA_Q_MEAS / (DELTA_CTX + 1e-8)
QPR_INPUT = DELTA_Q_MEAS / (DELTA_Q_INPUT + 1e-8)

QMRS_LIFT = QMRS_AP / POISON_RATE
RAW_LIFT = RAW_AP / POISON_RATE
CTX_LIFT = CTX_AP / POISON_RATE

# -----------------------------
# Tables
# -----------------------------
input_boundary_df = pd.DataFrame([
    {
        "Version": "V23",
        "Trigger": "Random global blend",
        "ASR": 0.9985,
        "CA": 0.7556,
        "QPR": 0.2863,
        "QMRS AUPRC": 1.0000,
        "Classical AUPRC": 1.0000,
        "Decision": "Valid attack but classically obvious",
    },
    {
        "Version": "V26",
        "Trigger": "HSV/Qcolor-inspired hue trigger",
        "ASR": 0.9955,
        "CA": 0.7307,
        "QPR": 0.1260,
        "QMRS AUPRC": 0.1972,
        "Classical AUPRC": 1.0000,
        "Decision": "Worse than anchor; classical dominates",
    },
    {
        "Version": "V27",
        "Trigger": "Proxy-optimized sinusoidal",
        "ASR": 0.9568,
        "CA": 0.7027,
        "QPR": 0.1256,
        "QMRS AUPRC": "0.1585–0.2363",
        "Classical AUPRC": "0.1667–0.2048",
        "Decision": "No viable quantum-preferential trigger",
    },
])

v28_primary_df = pd.DataFrame([
    {"Metric": "Branch", "Value": selected.get("branch"), "Gate": "latent/VQC-input branch", "Status": "Pass"},
    {"Metric": "Threat model", "Value": selected.get("threat_model"), "Gate": "white-box latent label required", "Status": "Pass"},
    {"Metric": "Candidate", "Value": selected.get("candidate"), "Gate": "selected candidate", "Status": "Pass"},
    {"Metric": "Alpha", "Value": selected.get("alpha"), "Gate": "lowest passing alpha", "Status": "Pass"},
    {"Metric": "ASR", "Value": ASR, "Gate": ">= 0.80", "Status": "Pass" if ASR >= 0.80 else "Fail"},
    {"Metric": "Clean Accuracy", "Value": CA, "Gate": ">= 0.60", "Status": "Pass" if CA >= 0.60 else "Fail"},
    {"Metric": "delta_pixel", "Value": DELTA_PIXEL, "Gate": "≈ 0", "Status": "Pass" if abs(DELTA_PIXEL) <= 1e-6 else "Fail"},
    {"Metric": "delta_ctx", "Value": DELTA_CTX, "Gate": "≈ 0", "Status": "Pass" if abs(DELTA_CTX) <= 1e-4 else "Fail"},
    {"Metric": "delta_q_input", "Value": DELTA_Q_INPUT, "Gate": "positive", "Status": "Pass"},
    {"Metric": "delta_q_measurement", "Value": DELTA_Q_MEAS, "Gate": "positive", "Status": "Pass" if DELTA_Q_MEAS > 0 else "Fail"},
    {"Metric": "QMRS AUPRC", "Value": QMRS_AP, "Gate": "> raw/context", "Status": "Pass" if QMRS_AP > max(RAW_AP, CTX_AP) else "Fail"},
    {"Metric": "QMRS F1", "Value": QMRS_F1, "Gate": "> raw/context", "Status": "Pass" if QMRS_F1 > max(RAW_F1, CTX_F1) else "Fail"},
])

lane_a_df = pd.DataFrame([
    {
        "Detector": "QMRS",
        "Type": "Quantum unsupervised",
        "Feature Space": "VQC measurements R^8",
        "AUPRC": QMRS_AP,
        "F1": QMRS_F1,
        "Score Direction": selected.get("qmrs_direction", "forward"),
        "Claim Role": "Main Lane A detector",
    },
    {
        "Detector": "raw_m",
        "Type": "Classical unsupervised",
        "Feature Space": "Pixels R^(3x28x28)",
        "AUPRC": RAW_AP,
        "F1": RAW_F1,
        "Score Direction": selected.get("raw_direction", "reversed"),
        "Claim Role": "Lane A baseline",
    },
    {
        "Detector": "ctx_m",
        "Type": "Classical unsupervised",
        "Feature Space": "ResNet context R^512",
        "AUPRC": CTX_AP,
        "F1": CTX_F1,
        "Score Direction": selected.get("context_direction", "reversed"),
        "Claim Role": "Lane A baseline",
    },
])

mechanism_df = pd.DataFrame([
    {"Quantity": "delta_pixel", "Value": DELTA_PIXEL, "Meaning": "No raw-pixel trigger exists"},
    {"Quantity": "delta_ctx", "Value": DELTA_CTX, "Meaning": "No ResNet-context trigger exists"},
    {"Quantity": "delta_q_input", "Value": DELTA_Q_INPUT, "Meaning": "Direct VQC input perturbation"},
    {"Quantity": "delta_q_measurement", "Value": DELTA_Q_MEAS, "Meaning": "VQC measurement response"},
    {"Quantity": "QPR_ctx", "Value": QPR_CTX, "Meaning": "Effectively unbounded because delta_ctx = 0"},
    {"Quantity": "QPR_input", "Value": QPR_INPUT, "Meaning": "VQC measurement amplification over q_input trigger"},
    {"Quantity": "Measurement silhouette", "Value": MEAS_SIL, "Meaning": "Third-cluster separation in quantum measurement space"},
    {"Quantity": "Context silhouette", "Value": CTX_SIL, "Meaning": "No cluster separation in classical context space"},
])

input_boundary_df.to_csv(TABLE_DIR / "table_v28_input_space_boundary.csv", index=False)
v28_primary_df.to_csv(TABLE_DIR / "table_v28_primary_metrics.csv", index=False)
lane_a_df.to_csv(TABLE_DIR / "table_v28_lane_a_detector_comparison.csv", index=False)
mechanism_df.to_csv(TABLE_DIR / "table_v28_mechanism_evidence.csv", index=False)

# -----------------------------
# Figures
# -----------------------------
plt.figure(figsize=(8, 5))
x = np.arange(3)
width = 0.35
plt.bar(x - width / 2, [QMRS_AP, RAW_AP, CTX_AP], width, label="AUPRC")
plt.bar(x + width / 2, [QMRS_F1, RAW_F1, CTX_F1], width, label="F1")
plt.xticks(x, ["QMRS", "raw_m", "ctx_m"])
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.title("V28 Lane A: QMRS vs Raw/Context Classical Baselines")
plt.legend()
for i, v in enumerate([QMRS_AP, RAW_AP, CTX_AP]):
    plt.text(i - width / 2, min(v + 0.03, 1.02), f"{v:.3f}", ha="center", fontsize=9)
for i, v in enumerate([QMRS_F1, RAW_F1, CTX_F1]):
    plt.text(i + width / 2, min(v + 0.03, 1.02), f"{v:.3f}", ha="center", fontsize=9)
savefig(FIG_DIR / "fig1_v28_latent_qmrs_vs_classical.png")

plt.figure(figsize=(8, 5))
labels = ["delta_pixel", "delta_ctx", "delta_q_input", "delta_q_measurement"]
vals = [DELTA_PIXEL, DELTA_CTX, DELTA_Q_INPUT, DELTA_Q_MEAS]
plt.bar(labels, vals)
plt.ylabel("Mean L2 shift")
plt.title("V28 Mechanism: Pixel/Context Zero, Measurement Shift Positive")
for i, v in enumerate(vals):
    plt.text(i, v + max(vals) * 0.03, f"{v:.4f}", ha="center", fontsize=9)
savefig(FIG_DIR / "fig2_v28_pixel_ctx_measurement_delta.png")

plt.figure(figsize=(8, 5))
labels = ["QMRS", "raw_m", "ctx_m"]
vals = [QMRS_LIFT, RAW_LIFT, CTX_LIFT]
plt.bar(labels, vals)
plt.ylabel("AUPRC / poison-rate baseline")
plt.title("V28 Detection Lift Over 10% Poison-Rate Baseline")
for i, v in enumerate(vals):
    plt.text(i, v + max(vals) * 0.03, f"{v:.1f}x", ha="center", fontsize=9)
savefig(FIG_DIR / "fig3_v28_detection_lift.png")

plt.figure(figsize=(7, 5))
labels = ["Measurement space", "Context space"]
vals = [MEAS_SIL, CTX_SIL]
plt.bar(labels, vals)
plt.axhline(0, linewidth=1)
plt.ylabel("Silhouette score")
plt.title("V28 Third-Cluster Evidence: Measurement vs Context")
for i, v in enumerate(vals):
    plt.text(i, v + (0.03 if v >= 0 else -0.06), f"{v:.3f}", ha="center", fontsize=9)
savefig(FIG_DIR / "fig4_v28_measurement_vs_context_silhouette.png")

plt.figure(figsize=(9, 5))
labels = ["V23 anchor", "V26 hue", "V27 sinusoidal"]
qpr_vals = [0.2863, 0.1260, 0.1256]
plt.bar(labels, qpr_vals)
plt.axhline(0.5726, linestyle="--", label="Input-space QPR gate")
plt.ylabel("Input-space QPR")
plt.title("Input-Space Blend Failed QPR Gate; V28 Uses Separate Latent Threat Model")
plt.legend()
for i, v in enumerate(qpr_vals):
    plt.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=9)
savefig(FIG_DIR / "fig5_v28_input_boundary_qpr.png")

# -----------------------------
# Markdown thesis section
# -----------------------------
lines = []

lines.append("# Blend/DermaMNIST Branch: Negative Boundary and Latent Quantum-Pathway Detection")
lines.append("")
lines.append("## Overview")
lines.append("")
lines.append("This section reports the Blend/DermaMNIST branch of QMedShield. It produced two findings:")
lines.append("")
lines.append("1. **Input-space Blend negative boundary:** V23–V27 showed that pixel-space Blend attacks can be attack-valid, but cannot produce quantum-preferential detection in this ResNet-18 + VQC architecture.")
lines.append("2. **VQC-input Blend positive latent result:** V28 injects a white-box latent trigger after the compressor and before the VQC, producing a strong QMRS Lane A advantage while leaving raw pixels and ResNet context unchanged.")
lines.append("")
lines.append("> Honest label: **VQC-Input Blend: a white-box latent backdoor injected into the 8-dimensional VQC input space after classical feature extraction.**")
lines.append("")
lines.append("This is not input-space Blend and does not authorize input-space Blend Notebook 2.")
lines.append("")
lines.append("## Input-Space Blend Boundary Evidence")
lines.append("")
lines.append(md_table(input_boundary_df))
lines.append("")
lines.append("Conclusion: input-space Blend is retained as a negative boundary branch, not the positive QNN/QMRS claim.")
lines.append("")
lines.append("## V28 VQC-Input Blend Primary Metrics")
lines.append("")
lines.append(md_table(v28_primary_df))
lines.append("")
lines.append("## Lane A Detector Comparison")
lines.append("")
lines.append(md_table(lane_a_df))
lines.append("")
lines.append("## Mechanism Evidence")
lines.append("")
lines.append(md_table(mechanism_df))
lines.append("")
lines.append("## Selected Candidate")
lines.append("")
lines.append(f"- Candidate: `{selected.get('candidate')}`")
lines.append(f"- Branch: `{selected.get('branch')}`")
lines.append(f"- Threat model: `{selected.get('threat_model')}`")
lines.append(f"- Alpha: `{selected.get('alpha')}`")
lines.append(f"- ASR: `{ASR:.4f}`")
lines.append(f"- CA: `{CA:.4f}`")
lines.append(f"- QMRS AUPRC/F1: `{QMRS_AP:.4f} / {QMRS_F1:.4f}`")
lines.append(f"- Raw AUPRC/F1: `{RAW_AP:.4f} / {RAW_F1:.4f}`")
lines.append(f"- Context AUPRC/F1: `{CTX_AP:.4f} / {CTX_F1:.4f}`")
lines.append("")
lines.append("## Claim Decision")
lines.append("")
lines.append("The V28 branch supports a positive **latent/VQC-input** claim:")
lines.append("")
lines.append("> The input-space Blend branch failed to produce quantum-preferential detection because its trigger entered through the ResNet context bottleneck. In contrast, the V28 latent/VQC-input Blend branch injects the trigger directly into the quantum input pathway, producing a measurement-space anomaly with zero pixel/context shift. Under the matched unsupervised Lane A protocol, QMRS detects the latent trigger better than raw-pixel and ResNet-context baselines. This supports a mechanistic quantum-pathway detection claim, not an input-space Blend superiority claim.")
lines.append("")
lines.append("## Limitations")
lines.append("")
lines.append("1. V28 is a white-box latent threat model.")
lines.append("2. It is not input-space Blend.")
lines.append("3. Lane B supervised probes are audit only.")
lines.append("4. The result is architecture-specific to the tested ResNet-18 + compressor + 8-qubit VQC pipeline.")
lines.append("5. Input-space Blend remains a negative boundary.")
lines.append("")
lines.append("## Generated Files")
lines.append("")
lines.append("- `outputs_blend_dermamnist/paper_tables/`")
lines.append("- `outputs_blend_dermamnist/paper_figures/`")
lines.append("- `Blend_DermaMNIST_V28_PAPER_EXPORT.zip`")

thesis_md = "\n".join(lines)

md_path = OUT / "blend_v28_thesis_section.md"
md_path.write_text(thesis_md, encoding="utf-8")

# -----------------------------
# Claim summary
# -----------------------------
claim_summary = {
    "verdict": "positive_latent_vqc_input_blend_only",
    "allowed_claim": "QMRS detects VQC-input latent trigger better than matched unsupervised raw/context Lane A baselines.",
    "not_allowed": "Do not claim input-space Blend QNN superiority.",
    "selected": selected,
}

claim_path = OUT / "blend_v28_claim_decision_summary.md"
claim_path.write_text(
    "# V28 Claim Decision Summary\n\n"
    + "```json\n"
    + json.dumps(claim_summary, indent=2)
    + "\n```\n",
    encoding="utf-8"
)

json_path = OUT / "blend_v28_selected_trigger_config_export.json"
json_path.write_text(json.dumps(selected, indent=2), encoding="utf-8")

# -----------------------------
# ZIP export
# -----------------------------
zip_path = Path("Blend_DermaMNIST_V28_PAPER_EXPORT.zip")

include_items = [
    md_path,
    claim_path,
    json_path,
    TABLE_DIR,
    FIG_DIR,
]

optional_outputs = [
    "blend_v28_latent_candidate_results.csv",
    "blend_v28_lane_a_unsupervised.csv",
    "blend_v28_lane_b_supervised_audit.csv",
    "blend_v28_qxai_observable_shift.csv",
    "blend_v28_qxai_score_correlation.csv",
    "blend_v28_qxai_mechanism_report.md",
    "blend_v28_claim_decision.md",
    "blend_v28_input_space_boundary_summary.md",
]

for name in optional_outputs:
    p = OUT / name
    if p.exists():
        include_items.append(p)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for item in include_items:
        item = Path(item)
        if item.is_file():
            z.write(item, arcname=str(item))
        elif item.is_dir():
            for f in item.rglob("*"):
                if f.is_file():
                    z.write(f, arcname=str(f))

print("✅ V28 paper export completed.")
print("Markdown:", md_path)
print("Claim summary:", claim_path)
print("Tables:", TABLE_DIR)
print("Figures:", FIG_DIR)
print("ZIP:", zip_path.resolve())

✅ V28 paper export completed.
Markdown: outputs_blend_dermamnist/blend_v28_thesis_section.md
Claim summary: outputs_blend_dermamnist/blend_v28_claim_decision_summary.md
Tables: outputs_blend_dermamnist/paper_tables
Figures: outputs_blend_dermamnist/paper_figures
ZIP: /content/Blend_DermaMNIST_V28_PAPER_EXPORT.zip
